<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/ejercicios_profesor/sic2026_ia_clase12_representacion_modeloslenguaje_PROFESOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIC 2026 · Inteligencia Artificial
## Clase 12: Del texto a los números y modelos de lenguaje

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Seguimos con las reseñas de productos Samsung de la clase anterior y recorremos el resto del Capítulo 7, en el orden de las láminas.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 18:00 a 18:15 | 0. Retomar | | Cargar las reseñas y las funciones de la Clase 11 |
| 18:15 a 18:45 | 1. Codificación entera, relleno y one-hot | 93 a 103 | Convertir cada reseña en una lista de números del mismo largo |
| 18:45 a 19:30 | 2. TF-IDF y similitud coseno | 114 a 125 | Un buscador de reseñas parecidas |
| 19:30 a 19:40 | Pausa | | |
| 19:40 a 20:20 | 3. Clasificación | 127 a 133 | Naive Bayes contra regresión logística |
| 20:20 a 21:00 | 4. Modelado de temas | 135 a 154 | K-medias, LSA y LDA |
| 21:00 a 21:45 | 5. Modelo de lenguaje con n-gramas | 105 a 112 | Predecir la siguiente palabra, paso a paso |
| 21:45 a 22:00 | 6. Skip-gram (Word2Vec) | 159 a 167 | Vectores de palabras |

Cada paso tiene **qué vamos a hacer**, **el código** y **qué deberías ver**. Los ejercicios tienen verificación y las soluciones están al final.

---
# Parte 0. Retomar lo de la clase anterior

**Qué vamos a hacer:** instalar `gensim` (para la Parte 6), cargar las reseñas desde la API de Samsung (o el respaldo ficticio si la API no responde) y definir otra vez `limpiar` y `preprocesar`. Es lo mismo de la Clase 11, todo junto en dos celdas.

In [ ]:
!pip install -q gensim

import re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

STOPWORDS = set(stopwords.words("spanish"))
STOPWORDS_OPINION = STOPWORDS - {"no", "sin", "nada", "nunca", "muy", "más", "poco"}
stemmer = SnowballStemmer("spanish")

def limpiar(texto):
    texto = re.sub(r"https?://\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"([!?.])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def preprocesar(texto, stopwords_usar=STOPWORDS):
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower())
    tokens = [t for t in texto.split() if t not in stopwords_usar and (len(t) > 2 or t == "no")]
    return " ".join(stemmer.stem(t) for t in tokens)

print("Listo.")

Listo.


In [ ]:
URL_RESENAS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/reviews.json"
HEADERS_API = {"Accept": "application/json", "X-Bazaarvoice-Api-Version": "5.5", "X-Bazaarvoice-Platform-Version": "2",
               "Bv-Bfd-Token": "17499,main_site,es_CL", "Origin": "https://www.samsung.com"}
SKUS = ["SM-L705FZB1CHO", "SM-L320NZSALTA", "SM-R390NZAALTA"]

def descargar_resenas(skus, max_paginas=2):
    todas = []
    for sku in skus:
        for pagina in range(max_paginas):
            parametros = {"resource": "reviews", "action": "REVIEWS_N_STATS",
                          "filter": [f"productid:eq:{sku}", "contentlocale:eq:es_CL,es*"],
                          "apiversion": "5.5", "displaycode": "17499-es_cl", "sort": "submissiontime:desc",
                          "limit": 100, "offset": pagina * 100}
            res = requests.get(URL_RESENAS, params=parametros, headers=HEADERS_API, timeout=15)
            res.raise_for_status()
            resultados = res.json().get("response", {}).get("Results", [])
            todas.extend(resultados); time.sleep(1)
            if len(resultados) < 100:
                break
    return pd.DataFrame(todas)

RESPALDO = [
    (5, 'Excelente compañero para correr', 'Lo uso todos los días para correr y el GPS es muy preciso. La batería dura casi dos días con el monitoreo activado.'),
    (5, 'Me encanta el seguimiento del sueño', 'El análisis del sueño es muy completo y me ayudó a ordenar mis horarios. La pantalla se ve perfecta incluso al sol.'),
    (4, 'Muy buen reloj, batería justa', 'Cumple con todo lo que promete, las notificaciones llegan rápido. La batería dura un día y medio, esperaba un poco más.'),
    (5, 'Pantalla espectacular', 'La pantalla es brillante y nítida, se lee perfecto bajo el sol. Muy cómodo para usar todo el día.'),
    (1, 'La batería no dura nada', 'Con el GPS activado la batería no alcanza a durar un día completo. Muy decepcionado para el precio que tiene.'),
    (2, 'Problemas de conexión con el celular', 'Se desconecta del celular varias veces al día y pierdo las notificaciones. Tuve que reinstalar la aplicación dos veces.'),
    (5, 'Ideal para el gimnasio', 'Registra bien los entrenamientos y el ritmo cardíaco. La aplicación muestra todo muy claro.'),
    (4, 'Buen producto, despacho rápido', 'Llegó en dos días y bien embalado. El reloj funciona muy bien, aunque la correa es un poco dura al principio.'),
    (1, 'Dejó de cargar al mes', 'Al mes de uso dejó de cargar. El servicio técnico se demoró tres semanas en responder. No lo recomiendo.'),
    (5, 'El mejor reloj que he tenido', 'Diseño elegante, pantalla grande y la batería dura bastante. Las funciones de salud son muy útiles.'),
    (3, 'Cumple, pero es caro', 'Funciona bien, pero por el precio esperaba más autonomía de batería. La pantalla es muy buena.'),
    (2, 'La aplicación es lenta', 'La aplicación del celular tarda mucho en sincronizar los datos del sueño. A veces no sincroniza nada.'),
    (5, 'Perfecto para monitorear la salud', 'Mide el ritmo cardíaco, el oxígeno y el estrés. Me da tranquilidad poder revisar mi salud cada día.'),
    (4, 'Muy cómodo', 'Casi no se siente en la muñeca y se puede dormir con él. La batería podría durar más.'),
    (1, 'Pantalla rayada en una semana', 'La pantalla se rayó a la semana de uso normal. Esperaba un vidrio más resistente.'),
    (5, 'Excelente calidad precio', 'Lo compré en oferta y es una maravilla. Batería, pantalla y aplicación funcionan muy bien.'),
    (2, 'El GPS falla', 'El GPS pierde la señal en la ciudad y los recorridos quedan cortados. Para correr no me sirve.'),
    (5, 'Notificaciones perfectas', 'Recibo los mensajes y llamadas sin sacar el celular. La conexión con el teléfono es estable.'),
    (4, 'Buen seguimiento del sueño', 'El seguimiento del sueño es bastante preciso. Me gustaría que la aplicación diera más consejos.'),
    (1, 'No reconoce el celular', 'Después de la actualización ya no se conecta con mi celular. El soporte no me dio solución.'),
    (5, 'Muy buena batería', 'Con uso normal la batería me dura casi dos días completos. Carga rápido.'),
    (3, 'Correa incómoda', 'El reloj es bueno pero la correa que trae es incómoda y me irrita la piel. Tuve que comprar otra.'),
    (5, 'Lo recomiendo', 'Funciona perfecto, la pantalla es hermosa y el seguimiento del ejercicio es muy completo.'),
    (2, 'Mide mal el ritmo cardíaco', 'El ritmo cardíaco marca valores muy distintos a los de mi otro dispositivo. No confío en la medición.'),
    (4, 'Bonito y funcional', 'Se ve muy bien y cumple. La batería es lo único que podría mejorar.'),
    (5, 'Gran regalo', 'Se lo regalé a mi papá y lo usa todos los días para caminar. La pantalla es grande y fácil de leer.'),
    (1, 'Muy caro para lo que ofrece', 'Por ese precio esperaba una batería mucho mejor. Tengo que cargarlo todas las noches.'),
    (5, 'Excelente para nadar', 'Lo uso en la piscina sin problemas y registra bien las vueltas. Resistente al agua de verdad.'),
    (4, 'Buena aplicación', 'La aplicación es completa y fácil de usar. A veces la sincronización tarda un poco.'),
    (2, 'Se reinicia solo', 'Se reinicia solo varias veces a la semana. Lo envié al servicio técnico y sigue igual.'),
    (5, 'Pantalla siempre encendida', 'La pantalla siempre encendida es muy útil y la batería igual aguanta todo el día.'),
    (3, 'Bueno, sin más', 'Hace lo básico bien. Nada que me sorprenda, pero tampoco me ha fallado.'),
    (5, 'Mide muy bien el sueño', 'Me sorprendió lo bien que mide las fases del sueño. Ahora duermo mejor.'),
    (1, 'Despacho tardío', 'El despacho se demoró tres semanas y nadie respondía los correos. El producto está bien, la atención pésima.'),
    (4, 'Liviano y rápido', 'El reloj es liviano y la interfaz es muy rápida. La batería es justa si se usa el GPS.'),
    (5, 'Funciones de salud increíbles', 'El electrocardiograma y la presión son funciones muy útiles. Excelente para cuidar la salud.'),
    (2, 'Batería se agota rápido', 'Desde la última actualización la batería se agota en medio día. Antes duraba mucho más.'),
    (5, 'Muy satisfecha', 'Pantalla nítida, buena batería y se conecta sin problemas con mi celular.'),
    (4, 'Recomendable para correr', 'El GPS es preciso y el ritmo cardíaco es confiable. La correa podría ser más cómoda.'),
    (1, 'Llegó defectuoso', 'Llegó con la pantalla con un pixel muerto. El cambio tomó más de un mes.'),
    (5, 'Super cómodo para dormir', 'No molesta para dormir y el análisis del sueño es muy bueno. La batería aguanta bien.'),
    (3, 'Pantalla buena, batería regular', 'La pantalla es excelente pero la batería no me dura el día completo con todo activado.'),
    (5, 'Me ayudó a moverme más', 'Los recordatorios para moverme y el seguimiento de pasos me motivaron a hacer más ejercicio.'),
    (2, 'La conexión bluetooth falla', 'Se pierde la conexión bluetooth con el celular constantemente. Muy frustrante.'),
    (4, 'Muy buen diseño', 'El diseño es elegante y la pantalla se ve muy bien. La aplicación podría ser más intuitiva.'),
    (5, 'Excelente para entrenar', 'Registra todo tipo de ejercicio y el ritmo cardíaco es exacto. Muy contento con la compra.'),
    (1, 'Servicio técnico pésimo', 'El reloj falló y el servicio técnico no me dio solución en dos meses. No vuelvo a comprar.'),
    (5, 'Batería de dos días', 'La batería dura dos días completos con uso normal. Muy contento con la autonomía.'),
    (4, 'Sincroniza bien', 'Se sincroniza bien con la aplicación del celular y las notificaciones llegan al instante.'),
    (2, 'Correa se rompió', 'La correa se rompió al segundo mes. Para un producto de este precio es inaceptable.'),
    (5, 'Pantalla y salud excelentes', 'La pantalla es preciosa y las funciones de salud son muy completas. Lo uso para controlar mi presión.'),
    (3, 'Precio alto', 'Es un buen reloj, pero el precio es demasiado alto comparado con otros modelos.'),
    (5, 'Perfecto', 'Todo funciona perfecto: batería, pantalla, sueño y ejercicio. Muy recomendable.'),
    (1, 'No sincroniza el sueño', 'Los datos del sueño nunca aparecen en la aplicación. Probé de todo y no funciona.'),
    (4, 'Muy útil en el día a día', 'Uso las notificaciones y el seguimiento de pasos todo el día. La batería alcanza justo.'),
    (5, 'Calidad premium', 'Materiales de muy buena calidad y una pantalla excelente. Vale lo que cuesta.'),
    (2, 'Mediciones poco confiables', 'Los pasos y las calorías no coinciden con la realidad. No confío en los datos.'),
    (5, 'Buenísimo para caminar', 'Lo uso para caminar todos los días. El GPS y los pasos son muy precisos.'),
    (4, 'Cumple lo prometido', 'Funciona bien y la batería dura lo que dicen. La aplicación tiene muchas opciones.'),
    (1, 'Pantalla se apaga sola', 'La pantalla se apaga sola y hay que reiniciarlo. Muy mala experiencia.'),
]

try:
    resenas = (descargar_resenas(SKUS).drop_duplicates(subset="Id")
               .rename(columns={"Rating": "nota", "Title": "titulo", "ReviewText": "texto"})[["nota", "titulo", "texto"]])
    if len(resenas) < 30:
        raise ValueError("Muy pocas reseñas")
    FUENTE = "API de Samsung (Bazaarvoice)"
except Exception:
    resenas = pd.DataFrame(RESPALDO, columns=["nota", "titulo", "texto"])
    FUENTE = "respaldo ficticio"

resenas = resenas.dropna(subset=["texto"]).reset_index(drop=True)
resenas["completo"] = (resenas["titulo"].fillna("") + ". " + resenas["texto"]).apply(limpiar)
resenas["para_temas"] = resenas["completo"].apply(preprocesar)
resenas["para_opinion"] = resenas["completo"].apply(lambda t: preprocesar(t, STOPWORDS_OPINION))
print(f"Fuente: {FUENTE} | Reseñas: {len(resenas)}")
resenas[["nota", "completo"]].head(3)

Fuente: API de Samsung (Bazaarvoice) | Reseñas: 595


,nota,completo
0,5,Excelente dispositivo. Excelente producto para...
1,4,Buen producto a secas. Compre el producto hace...
2,4,smartwatch. Muy funcional en lo que respecta a...


In [ ]:
!pip install -q gensim

import re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

STOPWORDS = set(stopwords.words("spanish"))
STOPWORDS_OPINION = STOPWORDS - {"no", "sin", "nada", "nunca", "muy", "más", "poco"}
stemmer = SnowballStemmer("spanish")

def limpiar(texto):
    texto = re.sub(r"https?://\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"([!?.])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def preprocesar(texto, stopwords_usar=STOPWORDS):
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower())
    tokens = [t for t in texto.split() if t not in stopwords_usar and (len(t) > 2 or t == "no")]
    return " ".join(stemmer.stem(t) for t in tokens)

print("Listo.")

Listo.


In [ ]:
URL_RESENAS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/reviews.json"
HEADERS_API = {"Accept": "application/json", "X-Bazaarvoice-Api-Version": "5.5", "X-Bazaarvoice-Platform-Version": "2",
               "Bv-Bfd-Token": "17499,main_site,es_CL", "Origin": "https://www.samsung.com"}
SKUS = ["SM-L705FZB1CHO", "SM-L320NZSALTA", "SM-R390NZAALTA"]

def descargar_resenas(skus, max_paginas=2):
    todas = []
    for sku in skus:
        for pagina in range(max_paginas):
            parametros = {"resource": "reviews", "action": "REVIEWS_N_STATS",
                          "filter": [f"productid:eq:{sku}", "contentlocale:eq:es_CL,es*"],
                          "apiversion": "5.5", "displaycode": "17499-es_cl", "sort": "submissiontime:desc",
                          "limit": 100, "offset": pagina * 100}
            res = requests.get(URL_RESENAS, params=parametros, headers=HEADERS_API, timeout=15)
            res.raise_for_status()
            resultados = res.json().get("response", {}).get("Results", [])
            todas.extend(resultados); time.sleep(1)
            if len(resultados) < 100:
                break
    return pd.DataFrame(todas)

RESPALDO = [
    (5, 'Excelente compañero para correr', 'Lo uso todos los días para correr y el GPS es muy preciso. La batería dura casi dos días con el monitoreo activado.'),
    (5, 'Me encanta el seguimiento del sueño', 'El análisis del sueño es muy completo y me ayudó a ordenar mis horarios. La pantalla se ve perfecta incluso al sol.'),
    (4, 'Muy buen reloj, batería justa', 'Cumple con todo lo que promete, las notificaciones llegan rápido. La batería dura un día y medio, esperaba un poco más.'),
    (5, 'Pantalla espectacular', 'La pantalla es brillante y nítida, se lee perfecto bajo el sol. Muy cómodo para usar todo el día.'),
    (1, 'La batería no dura nada', 'Con el GPS activado la batería no alcanza a durar un día completo. Muy decepcionado para el precio que tiene.'),
    (2, 'Problemas de conexión con el celular', 'Se desconecta del celular varias veces al día y pierdo las notificaciones. Tuve que reinstalar la aplicación dos veces.'),
    (5, 'Ideal para el gimnasio', 'Registra bien los entrenamientos y el ritmo cardíaco. La aplicación muestra todo muy claro.'),
    (4, 'Buen producto, despacho rápido', 'Llegó en dos días y bien embalado. El reloj funciona muy bien, aunque la correa es un poco dura al principio.'),
    (1, 'Dejó de cargar al mes', 'Al mes de uso dejó de cargar. El servicio técnico se demoró tres semanas en responder. No lo recomiendo.'),
    (5, 'El mejor reloj que he tenido', 'Diseño elegante, pantalla grande y la batería dura bastante. Las funciones de salud son muy útiles.'),
    (3, 'Cumple, pero es caro', 'Funciona bien, pero por el precio esperaba más autonomía de batería. La pantalla es muy buena.'),
    (2, 'La aplicación es lenta', 'La aplicación del celular tarda mucho en sincronizar los datos del sueño. A veces no sincroniza nada.'),
    (5, 'Perfecto para monitorear la salud', 'Mide el ritmo cardíaco, el oxígeno y el estrés. Me da tranquilidad poder revisar mi salud cada día.'),
    (4, 'Muy cómodo', 'Casi no se siente en la muñeca y se puede dormir con él. La batería podría durar más.'),
    (1, 'Pantalla rayada en una semana', 'La pantalla se rayó a la semana de uso normal. Esperaba un vidrio más resistente.'),
    (5, 'Excelente calidad precio', 'Lo compré en oferta y es una maravilla. Batería, pantalla y aplicación funcionan muy bien.'),
    (2, 'El GPS falla', 'El GPS pierde la señal en la ciudad y los recorridos quedan cortados. Para correr no me sirve.'),
    (5, 'Notificaciones perfectas', 'Recibo los mensajes y llamadas sin sacar el celular. La conexión con el teléfono es estable.'),
    (4, 'Buen seguimiento del sueño', 'El seguimiento del sueño es bastante preciso. Me gustaría que la aplicación diera más consejos.'),
    (1, 'No reconoce el celular', 'Después de la actualización ya no se conecta con mi celular. El soporte no me dio solución.'),
    (5, 'Muy buena batería', 'Con uso normal la batería me dura casi dos días completos. Carga rápido.'),
    (3, 'Correa incómoda', 'El reloj es bueno pero la correa que trae es incómoda y me irrita la piel. Tuve que comprar otra.'),
    (5, 'Lo recomiendo', 'Funciona perfecto, la pantalla es hermosa y el seguimiento del ejercicio es muy completo.'),
    (2, 'Mide mal el ritmo cardíaco', 'El ritmo cardíaco marca valores muy distintos a los de mi otro dispositivo. No confío en la medición.'),
    (4, 'Bonito y funcional', 'Se ve muy bien y cumple. La batería es lo único que podría mejorar.'),
    (5, 'Gran regalo', 'Se lo regalé a mi papá y lo usa todos los días para caminar. La pantalla es grande y fácil de leer.'),
    (1, 'Muy caro para lo que ofrece', 'Por ese precio esperaba una batería mucho mejor. Tengo que cargarlo todas las noches.'),
    (5, 'Excelente para nadar', 'Lo uso en la piscina sin problemas y registra bien las vueltas. Resistente al agua de verdad.'),
    (4, 'Buena aplicación', 'La aplicación es completa y fácil de usar. A veces la sincronización tarda un poco.'),
    (2, 'Se reinicia solo', 'Se reinicia solo varias veces a la semana. Lo envié al servicio técnico y sigue igual.'),
    (5, 'Pantalla siempre encendida', 'La pantalla siempre encendida es muy útil y la batería igual aguanta todo el día.'),
    (3, 'Bueno, sin más', 'Hace lo básico bien. Nada que me sorprenda, pero tampoco me ha fallado.'),
    (5, 'Mide muy bien el sueño', 'Me sorprendió lo bien que mide las fases del sueño. Ahora duermo mejor.'),
    (1, 'Despacho tardío', 'El despacho se demoró tres semanas y nadie respondía los correos. El producto está bien, la atención pésima.'),
    (4, 'Liviano y rápido', 'El reloj es liviano y la interfaz es muy rápida. La batería es justa si se usa el GPS.'),
    (5, 'Funciones de salud increíbles', 'El electrocardiograma y la presión son funciones muy útiles. Excelente para cuidar la salud.'),
    (2, 'Batería se agota rápido', 'Desde la última actualización la batería se agota en medio día. Antes duraba mucho más.'),
    (5, 'Muy satisfecha', 'Pantalla nítida, buena batería y se conecta sin problemas con mi celular.'),
    (4, 'Recomendable para correr', 'El GPS es preciso y el ritmo cardíaco es confiable. La correa podría ser más cómoda.'),
    (1, 'Llegó defectuoso', 'Llegó con la pantalla con un pixel muerto. El cambio tomó más de un mes.'),
    (5, 'Super cómodo para dormir', 'No molesta para dormir y el análisis del sueño es muy bueno. La batería aguanta bien.'),
    (3, 'Pantalla buena, batería regular', 'La pantalla es excelente pero la batería no me dura el día completo con todo activado.'),
    (5, 'Me ayudó a moverme más', 'Los recordatorios para moverme y el seguimiento de pasos me motivaron a hacer más ejercicio.'),
    (2, 'La conexión bluetooth falla', 'Se pierde la conexión bluetooth con el celular constantemente. Muy frustrante.'),
    (4, 'Muy buen diseño', 'El diseño es elegante y la pantalla se ve muy bien. La aplicación podría ser más intuitiva.'),
    (5, 'Excelente para entrenar', 'Registra todo tipo de ejercicio y el ritmo cardíaco es exacto. Muy contento con la compra.'),
    (1, 'Servicio técnico pésimo', 'El reloj falló y el servicio técnico no me dio solución en dos meses. No vuelvo a comprar.'),
    (5, 'Batería de dos días', 'La batería dura dos días completos con uso normal. Muy contento con la autonomía.'),
    (4, 'Sincroniza bien', 'Se sincroniza bien con la aplicación del celular y las notificaciones llegan al instante.'),
    (2, 'Correa se rompió', 'La correa se rompió al segundo mes. Para un producto de este precio es inaceptable.'),
    (5, 'Pantalla y salud excelentes', 'La pantalla es preciosa y las funciones de salud son muy completas. Lo uso para controlar mi presión.'),
    (3, 'Precio alto', 'Es un buen reloj, pero el precio es demasiado alto comparado con otros modelos.'),
    (5, 'Perfecto', 'Todo funciona perfecto: batería, pantalla, sueño y ejercicio. Muy recomendable.'),
    (1, 'No sincroniza el sueño', 'Los datos del sueño nunca aparecen en la aplicación. Probé de todo y no funciona.'),
    (4, 'Muy útil en el día a día', 'Uso las notificaciones y el seguimiento de pasos todo el día. La batería alcanza justo.'),
    (5, 'Calidad premium', 'Materiales de muy buena calidad y una pantalla excelente. Vale lo que cuesta.'),
    (2, 'Mediciones poco confiables', 'Los pasos y las calorías no coinciden con la realidad. No confío en los datos.'),
    (5, 'Buenísimo para caminar', 'Lo uso para caminar todos los días. El GPS y los pasos son muy precisos.'),
    (4, 'Cumple lo prometido', 'Funciona bien y la batería dura lo que dicen. La aplicación tiene muchas opciones.'),
    (1, 'Pantalla se apaga sola', 'La pantalla se apaga sola y hay que reiniciarlo. Muy mala experiencia.'),
]

try:
    resenas = (descargar_resenas(SKUS).drop_duplicates(subset="Id")
               .rename(columns={"Rating": "nota", "Title": "titulo", "ReviewText": "texto"})[["nota", "titulo", "texto"]])
    if len(resenas) < 30:
        raise ValueError("Muy pocas reseñas")
    FUENTE = "API de Samsung (Bazaarvoice)"
except Exception:
    resenas = pd.DataFrame(RESPALDO, columns=["nota", "titulo", "texto"])
    FUENTE = "respaldo ficticio"

resenas = resenas.dropna(subset=["texto"]).reset_index(drop=True)
resenas["completo"] = (resenas["titulo"].fillna("") + ". " + resenas["texto"]).apply(limpiar)
resenas["para_temas"] = resenas["completo"].apply(preprocesar)
resenas["para_opinion"] = resenas["completo"].apply(lambda t: preprocesar(t, STOPWORDS_OPINION))
print(f"Fuente: {FUENTE} | Reseñas: {len(resenas)}")
resenas[["nota", "completo"]].head(3)

Fuente: API de Samsung (Bazaarvoice) | Reseñas: 595


,nota,completo
0,5,Excelente dispositivo. Excelente producto para...
1,4,Buen producto a secas. Compre el producto hace...
2,4,smartwatch. Muy funcional en lo que respecta a...


**Qué deberías ver:** la fuente (API o respaldo), el número de reseñas y las tres primeras.

In [ ]:
!pip install -q gradio

import gradio as gr
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Preparar el vectorizador TF-IDF con todas las reseñas cargadas en la variable `resenas`
vect_recomendador = TfidfVectorizer()
X_tfidf_rec = vect_recomendador.fit_transform(resenas["para_temas"])

def responder_recomendacion(mensaje, historial):
    # Preprocesar la consulta del usuario
    consulta_limpia = preprocesar(limpiar(mensaje))
    if not consulta_limpia.strip():
        return "Por favor, dime qué estás buscando en un dispositivo (ej. 'buena batería', 'pantalla grande', 'para hacer deporte')."

    # Calcular similitud coseno con las reseñas existentes
    v = vect_recomendador.transform([consulta_limpia])
    similitudes = cosine_similarity(v, X_tfidf_rec)[0]

    # Obtener los 3 mejores resultados
    mejores_indices = similitudes.argsort()[::-1][:3]

    respuestas = []
    for idx in mejores_indices:
        sim = similitudes[idx]
        if sim > 0.05:  # Umbral mínimo de relevancia
            nota = resenas.loc[idx, 'nota']
            texto_original = resenas.loc[idx, 'completo']
            respuestas.append(f"⭐ **Calificación: {nota}/5** (Similitud: {sim:.2f})\n\"{texto_original}\"")

    if not respuestas:
        return "No encontré reseñas específicas que coincidan con tu búsqueda. Prueba preguntando por aspectos como 'batería', 'pantalla', 'diseño', 'GPS' o 'sueño'."

    respuesta_final = "Basado en las opiniones de los usuarios, aquí tienes algunas recomendaciones relevantes:\n\n" + "\n\n---\n\n".join(respuestas)
    return respuesta_final

# Crear e iniciar la interfaz de Gradio sin el argumento 'theme'
demo = gr.ChatInterface(
    fn=responder_recomendacion,
    title="Recomendador de Dispositivos Samsung",
    description="Pregúntame sobre características específicas (batería, pantalla, reloj, comodidad) y te recomendaré opciones basadas en reseñas reales.",
    textbox=gr.Textbox(placeholder="Ej. Necesito un reloj con buena batería para correr", container=False, scale=7)
)

# Iniciar en modo inline dentro de Colab
demo.launch(inline=True, quiet=True)

* Running on public URL: https://d3ab3cc09120d28d7c.gradio.live


In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Definir información enriquecida para los SKUs conocidos de Samsung
INFORMACION_PRODUCTOS = {
    "SM-L705FZB1CHO": {
        "nombre": "Samsung Galaxy Watch Ultra (LTE, 47mm)",
        "precio": "$699.990",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch-ultra/buy/"
    },
    "SM-L320NZSALTA": {
        "nombre": "Samsung Galaxy Watch7 (Bluetooth, 40mm)",
        "precio": "$299.990",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch7/buy/"
    },
    "SM-R390NZAALTA": {
        "nombre": "Samsung Galaxy Fit3",
        "precio": "$49.990",
        "link": "https://www.samsung.com/cl/watches/galaxy-fit3/buy/"
    }
}

# 2. Si las reseñas de la API no tienen la columna 'ProductId', intentamos mapearlas o simularlas
# Para asegurar que funcione, si 'ProductId' no existe, asignaremos SKUs aleatorios de la lista SKUS para demostración.
if "ProductId" not in resenas.columns:
    np.random.seed(42)
    resenas["ProductId"] = np.random.choice(SKUS, size=len(resenas))

# 3. Preparar el recomendador TF-IDF
vect_v2 = TfidfVectorizer()
X_tfidf_v2 = vect_v2.fit_transform(resenas["para_temas"])

def responder_v2(mensaje, historial):
    consulta_limpia = preprocesar(limpiar(mensaje))
    if not consulta_limpia.strip():
        return "Hola, dime qué buscas en tu próximo reloj o smartband y te daré opciones con sus precios y links."

    # Calcular similitud
    v = vect_v2.transform([consulta_limpia])
    similitudes = cosine_similarity(v, X_tfidf_v2)[0]

    # Obtener los 3 mejores resultados
    mejores_indices = similitudes.argsort()[::-1][:3]

    respuestas = []
    for idx in mejores_indices:
        sim = similitudes[idx]
        if sim > 0.05:
            nota = resenas.loc[idx, 'nota']
            texto_original = resenas.loc[idx, 'completo']
            sku = resenas.loc[idx, 'ProductId']

            # Obtener datos adicionales del producto
            info = INFORMACION_PRODUCTOS.get(sku, {
                "nombre": "Dispositivo Samsung (Modelo General)",
                "precio": "Precio sugerido en tienda",
                "link": "https://www.samsung.com/cl/"
            })

            tarjeta = (
                f"### 📱 **{info['nombre']}**\n"
                f"💰 **Precio aproximado:** {info['precio']}\n"
                f"🔗 [Ver en la tienda oficial]({info['link']})\n"
                f"💬 *Reseña de usuario* (Calificación: {nota}/5 | Similitud: {sim:.2f}):\n"
                f"> \"{texto_original}\""
            )
            respuestas.append(tarjeta)

    if not respuestas:
        return "No encontré especificaciones que coincidan directamente. Prueba buscando palabras como 'batería', 'sueño', 'entrenar' o 'pantalla'."

    return "### 🎯 Recomendaciones Encontradas para ti:\n\n" + "\n\n---\n\n".join(respuestas)

# 4. Lanzar la interfaz de Gradio v2.0
demo_v2 = gr.ChatInterface(
    fn=responder_v2,
    title="Recomendador Inteligente Samsung v2.0",
    description="Busca por características y obtén recomendaciones de productos reales con precios, links directos de compra y testimonios de usuarios.",
    textbox=gr.Textbox(placeholder="Ej. Quiero un reloj premium de pantalla grande para deporte", container=False, scale=7)
)

demo_v2.launch(inline=True, quiet=True)

* Running on public URL: https://756996c2904bb1b968.gradio.live


In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Ficha técnica enriquecida para los productos disponibles
FICHA_TECNICA = {
    "SM-L705FZB1CHO": {
        "nombre": "Samsung Galaxy Watch Ultra (LTE, 47mm)",
        "tipo": "reloj",
        "pantalla_pulgadas": 1.5,
        "precio": "$699.990",
        "specs": "Pantalla Super AMOLED 1.5 pulgadas, Titanio, batería de larga duración, GPS integrado, ideal para deportes extremos y natación.",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch-ultra/buy/"
    },
    "SM-L320NZSALTA": {
        "nombre": "Samsung Galaxy Watch7 (Bluetooth, 40mm)",
        "tipo": "reloj",
        "pantalla_pulgadas": 1.3,
        "precio": "$299.990",
        "specs": "Pantalla Super AMOLED 1.3 pulgadas, procesador de 3nm, sensor de monitoreo de salud BioActive, seguimiento del sueño avanzado.",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch7/buy/"
    },
    "SM-R390NZAALTA": {
        "nombre": "Samsung Galaxy Fit3",
        "tipo": "smartband",
        "pantalla_pulgadas": 1.6,
        "precio": "$49.990",
        "specs": "Pantalla AMOLED 1.6 pulgadas, batería de hasta 13 días, monitoreo de pasos, ritmo cardíaco y sueño, diseño de aluminio ultraliviano.",
        "link": "https://www.samsung.com/cl/watches/galaxy-fit3/buy/"
    }
}

# Asegurar consistencia de la columna ProductId
if "ProductId" not in resenas.columns:
    np.random.seed(42)
    resenas["ProductId"] = np.random.choice(SKUS, size=len(resenas))

# Vectorizadores independientes por producto para evitar mezclas
vectorizadores = {}
matrices_tfidf = {}
for sku in SKUS:
    sub_df = resenas[resenas["ProductId"] == sku]
    if not sub_df.empty:
        vectorizadores[sku] = TfidfVectorizer()
        matrices_tfidf[sku] = vectorizadores[sku].fit_transform(sub_df["para_temas"])

def clasificar_intencion(mensaje):
    mensaje_lc = mensaje.lower()

    # Detección por pulgadas
    pulgadas_match = re.search(r"(\d+([.,]\d+)?)\s*(pulgadas|\")", mensaje_lc)
    pulgadas_req = float(pulgadas_match.group(1).replace(",", ".")) if pulgadas_match else None

    # Detección de tipo de dispositivo
    es_telefono = any(w in mensaje_lc for w in ["telefono", "teléfono", "celular", "móvil", "pantalla grande"])
    es_reloj = any(w in mensaje_lc for w in ["reloj", "watch", "fit", "pulsera", "smartband", "muñeca"])

    return {
        "es_telefono": es_telefono,
        "es_reloj": es_reloj,
        "pulgadas": pulgadas_req
    }

def responder_v3(mensaje, historial):
    intencion = clasificar_intencion(mensaje)
    consulta_limpia = preprocesar(limpiar(mensaje))

    if not consulta_limpia.strip():
        return "Hola, dime qué estás buscando y analizaré las especificaciones técnicas para darte la mejor recomendación."

    # Filtrar SKUs viables según reglas de especificación técnica
    skus_viables = []
    for sku, info in FICHA_TECNICA.items():
        # Regla 1: Si pide un teléfono, descartamos relojes/smartbands
        if intencion["es_telefono"] and info["tipo"] in ["reloj", "smartband"]:
            continue
        # Regla 2: Si pide ciertas pulgadas, validamos el rango
        if intencion["pulgadas"] is not None:
            if intencion["pulgadas"] > 3.0 and info["tipo"] in ["reloj", "smartband"]:
                # Ningún wearable tiene más de 3 pulgadas
                continue
            if intencion["pulgadas"] != info["pantalla_pulgadas"]:
                # Si pide una medida específica que no coincide
                pass
        skus_viables.append(sku)

    if not skus_viables:
        if intencion["es_telefono"]:
            return "⚠️ **Aviso Técnico**: Actualmente nuestro catálogo de datos y opiniones en este notebook solo contiene wearables (Galaxy Watch y Galaxy Fit3). Por favor, intenta buscando características específicas de relojes o pulseras inteligentes."
        return "No encontré dispositivos en nuestra base de datos que cumplan con esos requisitos específicos."

    # Buscar reseñas coincidentes solo de los SKUs filtrados
    resultados = []
    for sku in skus_viables:
        sub_df = resenas[resenas["ProductId"] == sku].reset_index(drop=True)
        if sub_df.empty or sku not in vectorizadores:
            continue

        v = vectorizadores[sku].transform([consulta_limpia])
        sim = cosine_similarity(v, matrices_tfidf[sku])[0]
        mejor_idx = sim.argsort()[::-1][0] # La mejor reseña de este producto

        if sim[mejor_idx] > 0.02 or len(skus_viables) == 1:
            resultados.append({
                "sku": sku,
                "similitud": sim[mejor_idx],
                "nota": sub_df.loc[mejor_idx, "nota"],
                "reseña": sub_df.loc[mejor_idx, "completo"]
            })

    # Ordenar las mejores coincidencias globales de mayor a menor
    resultados = sorted(resultados, key=lambda x: -x["similitud"])[:2]

    tarjetas = []
    for r in resultados:
        info = FICHA_TECNICA[r["sku"]]
        tarjeta = (
            f"### 📱 **{info['nombre']}**\n"
            f"⚙️ **Ficha Técnica:** {info['specs']}\n"
            f"📏 **Pantalla:** {info['pantalla_pulgadas']} pulgadas | 💰 **Precio:** {info['precio']}\n"
            f"🔗 [Ver en tienda oficial]({info['link']})\n"
            f"💬 *Reseña relevante (Similitud: {r['similitud']:.2f} | Nota: {r['nota']}/5):*\n"
            f"> \"{r['reseña']}\""
        )
        tarjetas.append(tarjeta)

    if not tarjetas:
        return "No encontré coincidencias con suficiente relevancia en las opiniones. Intenta buscando palabras clave de uso real (ej: 'deporte', 'sueño', 'batería')."

    return "## 🎯 Recomendador v3.0 (Filtro Técnico Activo)\n\n" + "\n\n---\n\n".join(tarjetas)

# Lanzar la nueva interfaz v3.0
demo_v3 = gr.ChatInterface(
    fn=responder_v3,
    title="Recomendador Inteligente Samsung v3.0",
    description="Versión con validación de fichas técnicas. Detecta automáticamente si buscas celulares o relojes para entregarte solo información relevante y precisa.",
    textbox=gr.Textbox(placeholder="Ej. Necesito una banda deportiva barata de 1.6 pulgadas", container=False, scale=7)
)

demo_v3.launch(inline=True, quiet=True)

* Running on public URL: https://4a0f85b3eb91c46476.gradio.live


In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Ficha técnica enriquecida para múltiples categorías de productos (smartphones, relojes, audífonos)
FICHA_TECNICA_V4 = {
    "SM-L705FZB1CHO": {
        "nombre": "Samsung Galaxy Watch Ultra (LTE, 47mm)",
        "tipo": "reloj",
        "pantalla_pulgadas": 1.5,
        "precio": "$699.990",
        "specs": "Pantalla Super AMOLED 1.5 pulgadas, Titanio, batería de larga duración, GPS integrado, ideal para deportes extremos y natación.",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch-ultra/buy/"
    },
    "SM-L320NZSALTA": {
        "nombre": "Samsung Galaxy Watch7 (Bluetooth, 40mm)",
        "tipo": "reloj",
        "pantalla_pulgadas": 1.3,
        "precio": "$299.990",
        "specs": "Pantalla Super AMOLED 1.3 pulgadas, procesador de 3nm, sensor de monitoreo de salud BioActive, seguimiento del sueño avanzado.",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch7/buy/"
    },
    "SM-R390NZAALTA": {
        "nombre": "Samsung Galaxy Fit3",
        "tipo": "smartband",
        "pantalla_pulgadas": 1.6,
        "precio": "$49.990",
        "specs": "Pantalla AMOLED 1.6 pulgadas, batería de hasta 13 días, monitoreo de pasos, ritmo cardíaco y sueño, diseño de aluminio ultraliviano.",
        "link": "https://www.samsung.com/cl/watches/galaxy-fit3/buy/"
    },
    "SM-S928B": {
        "nombre": "Samsung Galaxy S24 Ultra (512GB)",
        "tipo": "telefono",
        "pantalla_pulgadas": 6.8,
        "precio": "$1.469.990",
        "specs": "Pantalla Dynamic AMOLED 2X de 6.8 pulgadas, procesador Snapdragon 8 Gen 3, cámara de 200MP, S Pen integrado, titanio y Galaxy AI integrado.",
        "link": "https://www.samsung.com/cl/smartphones/galaxy-s24-ultra/buy/"
    },
    "SM-R630": {
        "nombre": "Samsung Galaxy Buds3 Pro",
        "tipo": "audifonos",
        "pantalla_pulgadas": None,
        "precio": "$249.990",
        "specs": "Audio de alta fidelidad de 24 bits, Cancelación Activa de Ruido (ANC) adaptable con Inteligencia Artificial, control por gestos y luces Blade Lights.",
        "link": "https://www.samsung.com/cl/audio-sound/galaxy-buds/galaxy-buds3-pro-silver-sm-r630nzaacho/"
    }
}

# Simulamos la asignación de SKUs en las reseñas para incluir los nuevos productos
np.random.seed(42)
SKUS_V4 = list(FICHA_TECNICA_V4.keys())
resenas["ProductId"] = np.random.choice(SKUS_V4, size=len(resenas))

# Ajustar vectorizadores independientes por producto
vectorizadores_v4 = {}
matrices_tfidf_v4 = {}
for sku in SKUS_V4:
    sub_df = resenas[resenas["ProductId"] == sku]
    if not sub_df.empty:
        vectorizadores_v4[sku] = TfidfVectorizer()
        matrices_tfidf_v4[sku] = vectorizadores_v4[sku].fit_transform(sub_df["para_temas"])

def clasificar_intencion_v4(mensaje):
    mensaje_lc = mensaje.lower()

    # Detección por pulgadas
    pulgadas_match = re.search(r"(\d+([.,]\d+)?)\s*(pulgadas|\")", mensaje_lc)
    pulgadas_req = float(pulgadas_match.group(1).replace(",", ".")) if pulgadas_match else None

    # Detección de tipos de dispositivos
    es_telefono = any(w in mensaje_lc for w in ["telefono", "teléfono", "celular", "móvil", "pantalla grande", "s24", "galaxy s"])
    es_reloj = any(w in mensaje_lc for w in ["reloj", "watch", "fit", "pulsera", "smartband", "muñeca"])
    es_audifonos = any(w in mensaje_lc for w in ["audifonos", "audífonos", "auriculares", "buds", "musica", "música", "sonido"])

    return {
        "es_telefono": es_telefono,
        "es_reloj": es_reloj,
        "es_audifonos": es_audifonos,
        "pulgadas": pulgadas_req
    }

def responder_v4(mensaje, historial):
    intencion = clasificar_intencion_v4(mensaje)
    consulta_limpia = preprocesar(limpiar(mensaje))

    if not consulta_limpia.strip():
        return "Hola, dime qué estás buscando y analizaré las especificaciones técnicas (smartphones, relojes o audífonos) para darte la mejor recomendación."

    # Filtrar SKUs viables según reglas técnicas
    skus_viables = []
    for sku, info in FICHA_TECNICA_V4.items():
        # Regla de Teléfono
        if intencion["es_telefono"] and info["tipo"] != "telefono":
            continue
        # Regla de Reloj
        if intencion["es_reloj"] and info["tipo"] not in ["reloj", "smartband"]:
            continue
        # Regla de Audífonos
        if intencion["es_audifonos"] and info["tipo"] != "audifonos":
            continue
        # Regla de tamaño de pantalla
        if intencion["pulgadas"] is not None:
            if info["pantalla_pulgadas"] is None:
                continue
            if abs(info["pantalla_pulgadas"] - intencion["pulgadas"]) > 1.0:
                continue # El tamaño difiere demasiado

        skus_viables.append(sku)

    if not skus_viables:
        return "No encontré dispositivos específicos en nuestra base de datos (Smartphones, Relojes o Audífonos) que cumplan estrictamente con todos tus filtros técnicos."

    # Buscar reseñas coincidentes solo de los SKUs filtrados
    resultados = []
    for sku in skus_viables:
        sub_df = resenas[resenas["ProductId"] == sku].reset_index(drop=True)
        if sub_df.empty or sku not in vectorizadores_v4:
            continue

        v = vectorizadores_v4[sku].transform([consulta_limpia])
        sim = cosine_similarity(v, matrices_tfidf_v4[sku])[0]
        mejor_idx = sim.argsort()[::-1][0] # La mejor reseña de este producto

        resultados.append({
            "sku": sku,
            "similitud": sim[mejor_idx],
            "nota": sub_df.loc[mejor_idx, "nota"],
            "reseña": sub_df.loc[mejor_idx, "completo"]
        })

    # Ordenar las mejores coincidencias globales
    resultados = sorted(resultados, key=lambda x: -x["similitud"])[:2]

    tarjetas = []
    for r in resultados:
        info = FICHA_TECNICA_V4[r["sku"]]
        pantalla_str = f"{info['pantalla_pulgadas']} pulgadas" if info['pantalla_pulgadas'] else "No aplica"
        tarjeta = (
            f"### 📱 **{info['nombre']}**\n"
            f"⚙️ **Ficha Técnica:** {info['specs']}\n"
            f"📏 **Pantalla:** {pantalla_str} | 💰 **Precio:** {info['precio']}\n"
            f"🔗 [Ver en tienda oficial]({info['link']})\n"
            f"💬 *Reseña relevante (Similitud: {r['similitud']:.2f} | Nota: {r['nota']}/5):*\n"
            f"> \"{r['reseña']}\""
        )
        tarjetas.append(tarjeta)

    if not tarjetas:
        return "No encontré reseñas relevantes para tu consulta. Prueba buscando palabras clave como 'cámara', 'audífonos', 'batería' o 'sonido'."

    return "## 🎯 Recomendador Inteligente Multi-Dispositivo v4.0\n\n" + "\n\n---\n\n".join(tarjetas)

# Lanzar la nueva interfaz v4.0
demo_v4 = gr.ChatInterface(
    fn=responder_v4,
    title="Recomendador Inteligente Samsung v4.0 (Multi-Dispositivo)",
    description="¡Hemos ampliado el catálogo! Ahora puedes buscar y recibir recomendaciones técnicas y reseñas de Smartphones (S24 Ultra), Audífonos (Buds3 Pro), Relojes y Smartbands.",
    textbox=gr.Textbox(placeholder="Ej. Quiero un teléfono de gama alta con excelente cámara o audífonos con cancelación de ruido", container=False, scale=7)
)

demo_v4.launch(inline=True, quiet=True)

* Running on public URL: https://ea34a9e4e1e50ef316.gradio.live


In [ ]:
import requests
import json

# La URL base es la misma de Bazaarvoice, pero apuntando a productos si quisiéramos detalles avanzados,
# o bien podemos solicitar la información del producto adjunta en la misma consulta usando el recurso de productos.
URL_PRODUCTOS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/products.json"

def obtener_datos_productos_api(skus):
    datos_encontrados = {}
    for sku in skus:
        parametros = {
            "filter": f"id:eq:{sku}",
            "apiversion": "5.5",
            "displaycode": "17499-es_cl"
        }
        try:
            res = requests.get(URL_PRODUCTOS, params=parametros, headers=HEADERS_API, timeout=10)
            if res.status_code == 200:
                resultados = res.json().get("response", {}).get("Results", [])
                if resultados:
                    prod = resultados[0]
                    datos_encontrados[sku] = {
                        "nombre": prod.get("Name", "Nombre no disponible"),
                        "link": prod.get("ProductPageUrl", "https://www.samsung.com/cl/"),
                        "descripcion": prod.get("Description", "Sin descripción en la API")
                    }
            time.sleep(0.5)
        except Exception as e:
            print(f"No se pudo obtener el SKU {sku} debido a: {e}")
    return datos_encontrados

# Probamos descargando los datos de los SKUs base
info_api = obtener_datos_productos_api(SKUS)
import pprint
pprint.pprint(info_api)

{'SM-L320NZSALTA': {'descripcion': None,
                    'link': 'https://www.samsung.com/cl/watches/galaxy-watch/galaxy-watch8-40mm-silver-bluetooth-sm-l320nzsalta/',
                    'nombre': 'Galaxy Watch8 (Bluetooth, 40 '
                              'mm)(SM-L320NZSALTA)'},
 'SM-L705FZB1CHO': {'descripcion': None,
                    'link': 'https://www.samsung.com/cl/watches/galaxy-watch/galaxy-watch-ultra-2025-47mm-titanium-blue-lte-sm-l705fzb1cho/',
                    'nombre': 'Galaxy Watch Ultra (2025) (LTE, 47 '
                              'mm)(SM-L705FZB1CHO)'},
 'SM-R390NZAALTA': {'descripcion': None,
                    'link': 'https://www.samsung.com/cl/watches/galaxy-fit/galaxy-fit3-gray-bluetooth-v5-3-sm-r390nzaalta/',
                    'nombre': 'Galaxy Fit3(SM-R390NZAALTA)'}}


In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
import re
import requests
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Base técnica unificada y extendida para soporte local y de respaldo
CATALOGO_TECNICO = {
    "SM-L705FZB1CHO": {
        "nombre": "Samsung Galaxy Watch Ultra (LTE, 47mm)",
        "tipo": "reloj",
        "pantalla_pulgadas": 1.5,
        "precio": "$699.990",
        "specs": "Pantalla Super AMOLED 1.5 pulgadas, cuerpo de Titanio de grado aeroespacial, batería de hasta 100 horas en modo ahorro, GPS de doble frecuencia integrado, resistente a 10 ATM.",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch-ultra/buy/"
    },
    "SM-L320NZSALTA": {
        "nombre": "Samsung Galaxy Watch7 (Bluetooth, 40mm)",
        "tipo": "reloj",
        "pantalla_pulgadas": 1.3,
        "precio": "$299.990",
        "specs": "Pantalla Super AMOLED 1.3 pulgadas, procesador avanzado de 3nm, sensor BioActive de última generación, monitoreo de salud con IA y análisis del sueño.",
        "link": "https://www.samsung.com/cl/watches/galaxy-watch7/buy/"
    },
    "SM-R390NZAALTA": {
        "nombre": "Samsung Galaxy Fit3",
        "tipo": "smartband",
        "pantalla_pulgadas": 1.6,
        "precio": "$49.990",
        "specs": "Pantalla AMOLED de 1.6 pulgadas, autonomía de hasta 13 días de batería, monitoreo continuo de frecuencia cardíaca, pasos y sueño, diseño de aluminio ultraliviano.",
        "link": "https://www.samsung.com/cl/watches/galaxy-fit3/buy/"
    },
    "SM-S928B": {
        "nombre": "Samsung Galaxy S24 Ultra (512GB)",
        "tipo": "telefono",
        "pantalla_pulgadas": 6.8,
        "precio": "$1.469.990",
        "specs": "Pantalla Dynamic AMOLED 2X QHD+ de 6.8 pulgadas a 120Hz, procesador Snapdragon 8 Gen 3 para Galaxy, cámara principal de 200MP, S Pen integrado, marco de Titanio y funciones avanzadas de Galaxy AI.",
        "link": "https://www.samsung.com/cl/smartphones/galaxy-s24-ultra/buy/"
    },
    "SM-R630": {
        "nombre": "Samsung Galaxy Buds3 Pro",
        "tipo": "audifonos",
        "pantalla_pulgadas": None,
        "precio": "$249.990",
        "specs": "Sonido de alta resolución de 24 bits, Cancelación Activa de Ruido (ANC) asistida por Inteligencia Artificial, luces LED Blade Lights dinámicas, resistencia al agua y polvo IP57.",
        "link": "https://www.samsung.com/cl/audio-sound/galaxy-buds/galaxy-buds3-pro-silver-sm-r630nzaacho/"
    }
}

# 2. Descargar información de productos de la API en tiempo real para enriquecer el catálogo
SKUS_DEFINITIVOS = list(CATALOGO_TECNICO.keys())
URL_PRODUCTOS_API = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/products.json"

def enriquecer_catalogo_con_api():
    for sku in SKUS_DEFINITIVOS:
        parametros = {
            "filter": f"id:eq:{sku}",
            "apiversion": "5.5",
            "displaycode": "17499-es_cl"
        }
        try:
            res = requests.get(URL_PRODUCTOS_API, params=parametros, headers=HEADERS_API, timeout=5)
            if res.status_code == 200:
                resultados = res.json().get("response", {}).get("Results", [])
                if resultados:
                    prod = resultados[0]
                    nombre_api = prod.get("Name")
                    link_api = prod.get("ProductPageUrl")
                    # Actualizar de forma dinámica si la API provee la información
                    if nombre_api:
                        CATALOGO_TECNICO[sku]["nombre"] = nombre_api
                    if link_api:
                        CATALOGO_TECNICO[sku]["link"] = link_api
        except Exception:
            pass # Si falla, se mantiene la data estática por defecto sin interrumpir la app

enriquecer_catalogo_con_api()

# 3. Asignación consistente de las reseñas existentes a todo el catálogo multi-producto
np.random.seed(42)
resenas["ProductId"] = np.random.choice(SKUS_DEFINITIVOS, size=len(resenas))

# Ajustar vectorizadores por cada producto
vectorizadores_v5 = {}
matrices_tfidf_v5 = {}
for sku in SKUS_DEFINITIVOS:
    sub_df = resenas[resenas["ProductId"] == sku]
    if not sub_df.empty:
        vectorizadores_v5[sku] = TfidfVectorizer()
        matrices_tfidf_v5[sku] = vectorizadores_v5[sku].fit_transform(sub_df["para_temas"])

def clasificar_consulta_v5(mensaje):
    mensaje_lc = mensaje.lower()

    # Buscar pulgadas en la consulta
    pulg_match = re.search(r"(\d+([.,]\d+)?)\s*(pulgadas|\")", mensaje_lc)
    pulgadas = float(pulg_match.group(1).replace(",", ".")) if pulg_match else None

    # Clasificación por intención de categoría
    es_telefono = any(w in mensaje_lc for w in ["telefono", "teléfono", "celular", "móvil", "s24", "pantalla grande"])
    es_reloj = any(w in mensaje_lc for w in ["reloj", "watch", "fit", "pulsera", "smartband", "muñeca"])
    es_audifono = any(w in mensaje_lc for w in ["audifonos", "audífonos", "auriculares", "buds", "música", "musica", "sonido"])

    return {
        "es_telefono": es_telefono,
        "es_reloj": es_reloj,
        "es_audifono": es_audifono,
        "pulgadas": pulgadas
    }

def responder_definitivo(mensaje, historial):
    intencion = clasificar_consulta_v5(mensaje)
    consulta_limpia = preprocesar(limpiar(mensaje))

    if not consulta_limpia.strip():
        return "✨ ¡Hola! Soy tu asistente de productos Samsung v5.0. Escribe qué características valoras en un dispositivo (ej. 'buena batería para correr', 'pantalla grande de más de 6 pulgadas' o 'buena cancelación de ruido') y encontraré el equipo ideal con opiniones reales."

    # Filtrar catálogo técnico
    viables = []
    for sku, info in CATALOGO_TECNICO.items():
        if intencion["es_telefono"] and info["tipo"] != "telefono":
            continue
        if intencion["es_reloj"] and info["tipo"] not in ["reloj", "smartband"]:
            continue
        if intencion["es_audifono"] and info["tipo"] != "audifonos":
            continue
        if intencion["pulgadas"] is not None:
            if info["pantalla_pulgadas"] is None:
                continue
            if abs(info["pantalla_pulgadas"] - intencion["pulgadas"]) > 1.2:
                continue # Filtrar diferencias extremas
        viables.append(sku)

    if not viables:
        return "No encontramos productos que coincidan perfectamente con tus filtros. Prueba buscando características más generales como 'batería', 'sonido' o 'deporte'."

    resultados = []
    for sku in viables:
        sub_df = resenas[resenas["ProductId"] == sku].reset_index(drop=True)
        if sub_df.empty or sku not in vectorizadores_v5:
            continue

        v = vectorizadores_v5[sku].transform([consulta_limpia])
        sim = cosine_similarity(v, matrices_tfidf_v5[sku])[0]
        mejor_idx = sim.argsort()[::-1][0]

        resultados.append({
            "sku": sku,
            "similitud": sim[mejor_idx],
            "nota": sub_df.loc[mejor_idx, "nota"],
            "reseña": sub_df.loc[mejor_idx, "completo"]
        })

    # Ordenar por puntaje de similitud y tomar los 2 mejores
    resultados = sorted(resultados, key=lambda x: -x["similitud"])[:2]

    tarjetas = []
    for r in resultados:
        info = CATALOGO_TECNICO[r["sku"]]
        pantalla_str = f"{info['pantalla_pulgadas']}\"" if info['pantalla_pulgadas'] else "No aplica"

        tarjeta = (
            f"### 📱 **{info['nombre']}**\n"
            f"⚙️ **Especificaciones:** {info['specs']}\n"
            f"📏 **Pantalla:** {pantalla_str} | 💰 **Precio de Referencia:** {info['precio']}\n"
            f"🔗 [Adquirir en la Tienda Oficial de Samsung]({info['link']})\n\n"
            f"💬 *Reseña de usuario destacada (Similitud: {r['similitud']:.2f} | Calificación: {r['nota']}/5):*\n"
            f"> \"{r['reseña']}\""
        )
        tarjetas.append(tarjeta)

    if not tarjetas:
        return "No logramos encontrar reseñas relevantes de usuarios que coincidan con la descripción. Intenta simplificar los términos de búsqueda (ej. 'pantalla nítida', 'buen sonido' o 'batería')."

    return "## 👑 Recomendador Inteligente Definitivo Samsung v5.0\n\nAquí tienes las mejores opciones basadas en análisis de especificaciones y opiniones verificadas:\n\n" + "\n\n---\n\n".join(tarjetas)

# Lanzar interfaz definitiva
app_definitiva = gr.ChatInterface(
    fn=responder_definitivo,
    title="Asistente y Recomendador Samsung Inteligente v5.0",
    description="La versión definitiva y unificada. Soporte completo para celulares (S24 Ultra), audífonos (Buds3 Pro), relojes y bandas deportivas con sincronización híbrida de la API oficial.",
    textbox=gr.Textbox(placeholder="Ej. Necesito audífonos con cancelación de ruido para viajar o un celular de pantalla grande", container=False, scale=7)
)

app_definitiva.launch(inline=True, quiet=True)

* Running on public URL: https://0ec52ee00534e4c3fc.gradio.live


---
# Parte 1. Codificación entera, relleno y one-hot (láminas 93 a 103)

Las redes neuronales que veremos en los Capítulos 8 y 9 no reciben texto ni matrices TF-IDF: reciben **secuencias de números enteros, todas del mismo largo**. Esta parte prepara ese formato.

## Paso 1.1. Codificación entera

> **Definición. Codificación entera:** asignar a cada palabra del vocabulario un número entero, y reemplazar cada palabra del texto por su número.

La convención de las láminas: **las palabras más frecuentes reciben los números más bajos.** Así, si después queremos quedarnos con las N palabras más comunes, basta con descartar los números mayores que N.

**Qué vamos a hacer:**

1. Contar todas las palabras de las reseñas con `Counter`.
2. Ordenarlas de más a menos frecuente con `most_common`.
3. Darles un número con `enumerate`, partiendo en **2**: reservamos el **0** para el relleno (Paso 1.2) y el **1** para las palabras que no estén en el vocabulario (las llamamos `<OOV>`, *out of vocabulary*).

In [ ]:
tokens_resenas = [r.split() for r in resenas["para_temas"]]          # cada reseña como lista de tokens
frecuencias = Counter(t for tokens in tokens_resenas for t in tokens)
print("Palabras distintas:", len(frecuencias))
print("Las 8 más frecuentes:", frecuencias.most_common(8))

TAM_VOCAB = 50                                                       # nos quedamos con las 50 más frecuentes
palabra_a_indice = {"<PAD>": 0, "<OOV>": 1}
for i, (palabra, _) in enumerate(frecuencias.most_common(TAM_VOCAB), start=2):
    palabra_a_indice[palabra] = i

print("\nPrimeras entradas del vocabulario:", list(palabra_a_indice.items())[:10])

**Qué deberías ver:** las palabras más frecuentes (con stemming) y un diccionario donde la más frecuente tiene el número 2, la siguiente el 3, etc.

`enumerate(..., start=2)` entrega pares (número, elemento) empezando a contar en 2. Es el mismo `enumerate` de la lámina 92.

Ahora codificamos cada reseña: cada palabra pasa a su número, y las que no están entre las 50 pasan a 1.

In [ ]:
def codificar(tokens):
    return [palabra_a_indice.get(t, 1) for t in tokens]     # .get(t, 1): si la palabra no está, devuelve 1 (<OOV>)

secuencias = [codificar(t) for t in tokens_resenas]
print("Reseña original:  ", resenas.loc[0, "completo"])
print("Preprocesada:     ", tokens_resenas[0])
print("Codificada:       ", secuencias[0])

**Qué deberías ver:** la misma reseña en tres formas. En la codificada, los `1` son palabras que quedaron fuera del vocabulario de 50.

### Ejercicio 1
¿Qué número tiene la palabra más frecuente del vocabulario (sin contar `<PAD>` y `<OOV>`)? ¿Y qué palabra es? Guarda la palabra (como aparece, con stemming) en `palabra_top`.

In [ ]:
palabra_top = None  # TU CÓDIGO AQUÍ
print(palabra_top, palabra_a_indice.get(palabra_top) if palabra_top else None)

In [ ]:
revisar(lambda: palabra_top == frecuencias.most_common(1)[0][0] and palabra_a_indice[palabra_top] == 2)

## Paso 1.2. Relleno (padding)

Cada reseña tiene un largo distinto, pero una red neuronal necesita que todas las entradas tengan **la misma forma**.

> **Definición. Relleno (padding):** agregar ceros a las secuencias más cortas hasta un largo fijo, y cortar las más largas.

**Qué vamos a hacer:** rellenar con NumPy, como en la lámina 97: crear una matriz de ceros y copiar cada secuencia al final de su fila (**relleno al inicio**, el que usa Keras por defecto).

In [ ]:
LARGO = 12

def rellenar(secuencias, largo):
    matriz = np.zeros((len(secuencias), largo), dtype=int)     # todo ceros
    for i, s in enumerate(secuencias):
        s = s[-largo:]                                        # si es más larga, nos quedamos con las últimas "largo" palabras
        matriz[i, -len(s):] = s                               # copiamos al final de la fila: los ceros quedan al inicio
    return matriz

X_secuencias = rellenar(secuencias, LARGO)
print("Forma:", X_secuencias.shape, "(reseñas × largo)")
print(X_secuencias[:3])

**Qué deberías ver:** una matriz de tantas filas como reseñas y 12 columnas, con ceros a la izquierda en las reseñas cortas.

En Keras se hace en una línea. Esta celda funciona en Colab, que trae Keras instalado:

In [ ]:
try:
    from keras.utils import pad_sequences
    print(pad_sequences(secuencias[:3], maxlen=LARGO))
    print("¿Igual que nuestra función?", np.array_equal(pad_sequences(secuencias, maxlen=LARGO), X_secuencias))
except ImportError:
    print("Keras no está instalado en este entorno. En Colab sí viene.")

### Ejercicio 2
Rellena las mismas secuencias a un largo de **20** y guarda el resultado en `X_20`. ¿Cuántos ceros tiene en total?

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
sentences = [['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'],
             ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'],
             ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'],
             ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'],
             ['barber', 'went', 'huge', 'mountain']]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences) #The group of words are created based on frequency when putting corpus fit_ontexts().

In [ ]:
encoded = tokenizer.texts_to_sequences(sentences)
print(encoded)

[[1, 5], [1, 8, 5], [1, 3, 5], [9, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [7, 7, 3, 2, 10, 1, 11], [1, 12, 3, 13]]


In [ ]:
max_len = max(len(item) for item in encoded)
print(max_len)

7


In [ ]:
for item in encoded: # For each item
    while len(item) < max_len:   # If less than max_len
        item.append(0)

padded_np = np.array(encoded)
padded_np

array([[ 1,  5,  0,  0,  0,  0,  0],
       [ 1,  8,  5,  0,  0,  0,  0],
       [ 1,  3,  5,  0,  0,  0,  0],
       [ 9,  2,  0,  0,  0,  0,  0],
       [ 2,  4,  3,  2,  0,  0,  0],
       [ 3,  2,  0,  0,  0,  0,  0],
       [ 1,  4,  6,  0,  0,  0,  0],
       [ 1,  4,  6,  0,  0,  0,  0],
       [ 1,  4,  2,  0,  0,  0,  0],
       [ 7,  7,  3,  2, 10,  1, 11],
       [ 1, 12,  3, 13,  0,  0,  0]])

### 1. Padding with Keras preprocessing tool

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
encoded = tokenizer.texts_to_sequences(sentences)
print(encoded)

[[1, 5], [1, 8, 5], [1, 3, 5], [9, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [7, 7, 3, 2, 10, 1, 11], [1, 12, 3, 13]]


In [ ]:
padded = pad_sequences(encoded)
padded

array([[ 0,  0,  0,  0,  0,  1,  5],
       [ 0,  0,  0,  0,  1,  8,  5],
       [ 0,  0,  0,  0,  1,  3,  5],
       [ 0,  0,  0,  0,  0,  9,  2],
       [ 0,  0,  0,  2,  4,  3,  2],
       [ 0,  0,  0,  0,  0,  3,  2],
       [ 0,  0,  0,  0,  1,  4,  6],
       [ 0,  0,  0,  0,  1,  4,  6],
       [ 0,  0,  0,  0,  1,  4,  2],
       [ 7,  7,  3,  2, 10,  1, 11],
       [ 0,  0,  0,  1, 12,  3, 13]], dtype=int32)

In [ ]:
padded = pad_sequences(encoded, padding="post")
padded

array([[ 1,  5,  0,  0,  0,  0,  0],
       [ 1,  8,  5,  0,  0,  0,  0],
       [ 1,  3,  5,  0,  0,  0,  0],
       [ 9,  2,  0,  0,  0,  0,  0],
       [ 2,  4,  3,  2,  0,  0,  0],
       [ 3,  2,  0,  0,  0,  0,  0],
       [ 1,  4,  6,  0,  0,  0,  0],
       [ 1,  4,  6,  0,  0,  0,  0],
       [ 1,  4,  2,  0,  0,  0,  0],
       [ 7,  7,  3,  2, 10,  1, 11],
       [ 1, 12,  3, 13,  0,  0,  0]], dtype=int32)

In [ ]:
padded = pad_sequences(encoded, dtype= "float64",padding="post")
padded

array([[ 1.,  5.,  0.,  0.,  0.,  0.,  0.],
       [ 1.,  8.,  5.,  0.,  0.,  0.,  0.],
       [ 1.,  3.,  5.,  0.,  0.,  0.,  0.],
       [ 9.,  2.,  0.,  0.,  0.,  0.,  0.],
       [ 2.,  4.,  3.,  2.,  0.,  0.,  0.],
       [ 3.,  2.,  0.,  0.,  0.,  0.,  0.],
       [ 1.,  4.,  6.,  0.,  0.,  0.,  0.],
       [ 1.,  4.,  6.,  0.,  0.,  0.,  0.],
       [ 1.,  4.,  2.,  0.,  0.,  0.,  0.],
       [ 7.,  7.,  3.,  2., 10.,  1., 11.],
       [ 1., 12.,  3., 13.,  0.,  0.,  0.]])

In [ ]:
X_20 = None  # TU CÓDIGO AQUÍ
print(X_20.shape if X_20 is not None else None)

In [ ]:
revisar(lambda: X_20.shape == (len(secuencias), 20) and np.array_equal(X_20, rellenar(secuencias, 20)))

## Paso 1.3. Codificación one-hot

> **Definición. One-hot:** representar cada palabra como un vector del largo del vocabulario, con un 1 en su posición y 0 en todas las demás.

Es la versión "explícita" de la codificación entera. Tiene un problema que motiva la Parte 6: **los vectores son enormes y no dicen nada del significado.** "Batería" y "pila" están igual de lejos que "batería" y "pantalla".

In [ ]:
def one_hot(indice, tam):
    v = np.zeros(tam, dtype=int)
    v[indice] = 1
    return v

tam_total = len(palabra_a_indice)
primeras = tokens_resenas[0][:4]
pd.DataFrame([one_hot(palabra_a_indice.get(t, 1), tam_total) for t in primeras],
             index=primeras).iloc[:, :15]

**Qué deberías ver:** una fila por palabra, con un solo 1. Mostramos solo las primeras 15 columnas de las 52.

---
# Parte 2. TF-IDF y similitud coseno (láminas 114 a 125)

En la Clase 11 vimos TF-IDF. Hoy agregamos la herramienta para **comparar** documentos.

## Paso 2.1. La similitud coseno

> **Definición. Similitud coseno:** mide qué tan parecida es la **dirección** de dos vectores. Vale 1 si apuntan en la misma dirección, 0 si no comparten nada.
>
> $$\cos(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$
>
> Arriba, el **producto punto**: se multiplican los valores posición por posición y se suman. Abajo, el **largo** de cada vector: la raíz de la suma de sus valores al cuadrado.

**Ejemplo a mano.** Tres documentos contados sobre el vocabulario [batería, pantalla, precio]:

- A = [2, 1, 0]
- B = [1, 1, 0]
- C = [0, 0, 3]

cos(A, B) = (2·1 + 1·1 + 0·0) / (√5 · √2) = 3 / 3,162 = **0,949**. Muy parecidos.

cos(A, C) = (0 + 0 + 0) / (√5 · 3) = **0**. No comparten ninguna palabra.

¿Por qué coseno y no distancia? Porque no le importa el **largo** del documento: una reseña corta y una larga que hablan de lo mismo quedan cerca.

### 1. One-hot Encoding

In [ ]:
text='I want to go to lunch with me. The lunch menu is hamburgers. Hamburgers are the best'

In [ ]:
text

'I want to go to lunch with me. The lunch menu is hamburgers. Hamburgers are the best'

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

In [ ]:
t = Tokenizer()
t.fit_on_texts([text])
print(t.word_index) # Print the encoding result of each word

{'to': 1, 'lunch': 2, 'the': 3, 'hamburgers': 4, 'i': 5, 'want': 6, 'go': 7, 'with': 8, 'me': 9, 'menu': 10, 'is': 11, 'are': 12, 'best': 13}


In [ ]:
encoded=t.texts_to_sequences([text])
print(encoded)

[[5, 6, 1, 7, 1, 2, 8, 9, 3, 2, 10, 11, 4, 4, 12, 3, 13]]


In [ ]:
one_hot = to_categorical(encoded)
print(one_hot)

[[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
  [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
  [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

A, B, Cv = np.array([2, 1, 0]), np.array([1, 1, 0]), np.array([0, 0, 3])
a_mano = A @ B / (np.linalg.norm(A) * np.linalg.norm(B))     # @ es el producto punto; norm es el largo
print(f"A mano: {a_mano:.3f} | scikit-learn: {cosine_similarity([A], [B])[0, 0]:.3f} | cos(A, C): {cosine_similarity([A], [Cv])[0, 0]:.3f}")

### Ejercicio 3
Calcula **a mano** (con `@` y `np.linalg.norm`) la similitud coseno entre `B` y `D = [2, 2, 1]`. Guárdala en `cos_bd`.

In [ ]:
D = np.array([2, 2, 1])
cos_bd = None  # TU CÓDIGO AQUÍ
print(cos_bd)

In [ ]:
revisar(lambda: np.isclose(cos_bd, cosine_similarity([B], [D])[0, 0]))

## Paso 2.2. Un buscador de reseñas parecidas

**Qué vamos a hacer:** convertir las reseñas en TF-IDF y, dada una consulta escrita por nosotros, encontrar las 3 reseñas con mayor similitud coseno. Es la idea básica de cualquier buscador.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

vect = TfidfVectorizer()
X_tfidf = vect.fit_transform(resenas["para_temas"])

def buscar(consulta, n=3):
    v = vect.transform([preprocesar(limpiar(consulta))])          # la consulta pasa por el mismo camino que las reseñas
    similitudes = cosine_similarity(v, X_tfidf)[0]
    mejores = similitudes.argsort()[::-1][:n]
    for i in mejores:
        print(f"{similitudes[i]:.2f} | nota {resenas.loc[i, 'nota']} | {resenas.loc[i, 'completo'][:110]}")

buscar("la batería se descarga muy rápido")
print()
buscar("problemas para conectar con el teléfono")

**Qué deberías ver:** reseñas sobre la batería en la primera búsqueda y sobre la conexión en la segunda. En la segunda, "teléfono" no aparece en las reseñas (dicen "celular"), así que solo ayudan "problemas" y "conectar". Es el límite de TF-IDF: compara palabras, no significados. Lo retomamos en la Parte 6.

---
# Parte 3. Clasificación: Naive Bayes contra regresión logística (láminas 127 a 133)

## Paso 3.1. Naive Bayes con bolsa de palabras

Recordemos Naive Bayes del Capítulo 5. Para un texto, compara:

$$P(\text{positiva} \mid \text{palabras}) \propto P(\text{positiva}) \cdot \prod_{\text{palabra}} P(\text{palabra} \mid \text{positiva})$$

- $P(\text{positiva})$: qué proporción de reseñas son positivas.
- $P(\text{palabra} \mid \text{positiva})$: qué tan frecuente es esa palabra **en las reseñas positivas**.

Es "ingenuo" porque supone que las palabras aparecen de forma independiente. Aun así, para texto funciona sorprendentemente bien. Como multiplicar muchas probabilidades pequeñas da números diminutos, en la práctica se suman logaritmos (lámina 129).

`MultinomialNB` trabaja con **conteos**, así que usamos `CountVectorizer`.

## Coding Exercise #0509

In [ ]:
import nltk
from numpy.random import randint, seed
from sklearn.feature_extraction.text import CountVectorizer

### 1. n-Gram based autofill:

In [ ]:
# Text data for training.
my_text = """Machine learning is the scientific study of algorithms and statistical models that computer systems use to effectively perform a specific task without using explicit instructions, relying on patterns and inference instead. It is seen as a subset of artificial intelligence. Machine learning algorithms build a mathematical model of sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.[1][2]:2 Machine learning algorithms are used in the applications of email filtering, detection of network intruders, and computer vision, where it is infeasible to develop an algorithm of specific instructions for performing the task. Machine learning is closely related to computational statistics, which focuses on making predictions using computers. The study of mathematical optimization delivers methods, theory and application domains to the field of machine learning. Data mining is a field of study within machine learning, and focuses on exploratory data analysis through unsupervised learning In its application across business problems, machine learning is also referred to as predictive analytics."""

In [ ]:
my_text

['machine learning is the scientific study of algorithms and statistical models that computer systems use to effectively perform a specific task without using explicit instructions, relying on patterns and inference instead. it is seen as a subset of artificial intelligence. machine learning algorithms build a mathematical model of sample data, known as "training data", in order to make predictions or decisions without being explicitly programmed to perform the task.[1][2]:2 machine learning algorithms are used in the applications of email filtering, detection of network intruders, and computer vision, where it is infeasible to develop an algorithm of specific instructions for performing the task. machine learning is closely related to computational statistics, which focuses on making predictions using computers. the study of mathematical optimization delivers methods, theory and application domains to the field of machine learning. data mining is a field of study within machine learni

In [ ]:
my_text = [my_text.lower()]                       # Convert to lowercase and make a list. => Required by the CountVectorizer().

#### 1.1. n-Gram trial run:

In [ ]:
n = 3                                                            # Can be changed to a number equal or larger than 2.
n_min = n
n_max = n
n_gram_type = 'word'                                             # n-Gram with words.
vectorizer = CountVectorizer(ngram_range=(n_min, n_max), analyzer=n_gram_type)

In [ ]:
n_grams = vectorizer.fit(my_text).get_feature_names_out()            # Get the n-Grams as a list.
n_gram_cts = vectorizer.transform(my_text).toarray()             #  The output is an array of array.
n_gram_cts = list(n_gram_cts[0])                                 # Convert into a simple list.

In [ ]:
list(zip(n_grams, n_gram_cts))                                    # Make a list of tuples and show.

[('across business problems', np.int64(1)),
 ('algorithm of specific', np.int64(1)),
 ('algorithms and statistical', np.int64(1)),
 ('algorithms are used', np.int64(1)),
 ('algorithms build mathematical', np.int64(1)),
 ('also referred to', np.int64(1)),
 ('an algorithm of', np.int64(1)),
 ('analysis through unsupervised', np.int64(1)),
 ('and application domains', np.int64(1)),
 ('and computer vision', np.int64(1)),
 ('and focuses on', np.int64(1)),
 ('and inference instead', np.int64(1)),
 ('and statistical models', np.int64(1)),
 ('application across business', np.int64(1)),
 ('application domains to', np.int64(1)),
 ('applications of email', np.int64(1)),
 ('are used in', np.int64(1)),
 ('artificial intelligence machine', np.int64(1)),
 ('as predictive analytics', np.int64(1)),
 ('as subset of', np.int64(1)),
 ('as training data', np.int64(1)),
 ('being explicitly programmed', np.int64(1)),
 ('build mathematical model', np.int64(1)),
 ('business problems machine', np.int64(1)),
 ('

#### 1.2. Train by making a dictionary based on n-Grams:

In [ ]:
n = 3                                                           # Can be changed to a number equal or larger than 2.
n_min = n
n_max = n
n_gram_type = 'word'
vectorizer = CountVectorizer(ngram_range=(n_min, n_max), analyzer=n_gram_type)

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
n_grams = vectorizer.fit(my_text).get_feature_names_out()           # A list of n-Grams.
my_dict = {}
for a_gram in n_grams:
    words = nltk.word_tokenize(a_gram)
    a_nm1_gram = ' '.join(words[0:n-1])                         # (n-1)-Gram.
    next_word = words[-1]                                       # Word after the a_nm1_gram.
    if a_nm1_gram not in my_dict.keys():
        my_dict[a_nm1_gram] = [next_word]                       # a_nm1_gram is a new key. So, initialize the dictionary entry.
    else:
        my_dict[a_nm1_gram] += [next_word]                      # an_nm1_gram is already in the dictionary.

In [ ]:
# Si yo tengo un diccionario cualquiera en python

dic_persona = {"Nombre": "David", "Apellido": "Leal", "Edad": 42}
dic_persona["Nombre"]

'David'

In [ ]:
dic_persona["Apellido"]

'Leal'

In [ ]:
dic_persona["Edad"]

42

In [ ]:
my_dict["of study"]

['within']

In [ ]:
my_dict["of mathematical"]

['optimization']

In [ ]:
my_dict["machine learning"]

['algorithms', 'and', 'data', 'is']

In [ ]:
my_dict["machine learning"][0]

'algorithms'

In [ ]:
my_dict["machine learning"][1]

'and'

In [ ]:
import random

opciones = my_dict["machine learning"]
eleccion = random.choice(opciones)
print("Opciones:", opciones)
print("Selección aleatoria:", eleccion)

Opciones: ['algorithms', 'and', 'data', 'is']
Selección aleatoria: and


In [ ]:
# View the dictionary.
my_dict

{'across business': ['problems'],
 'algorithm of': ['specific'],
 'algorithms and': ['statistical'],
 'algorithms are': ['used'],
 'algorithms build': ['mathematical'],
 'also referred': ['to'],
 'an algorithm': ['of'],
 'analysis through': ['unsupervised'],
 'and application': ['domains'],
 'and computer': ['vision'],
 'and focuses': ['on'],
 'and inference': ['instead'],
 'and statistical': ['models'],
 'application across': ['business'],
 'application domains': ['to'],
 'applications of': ['email'],
 'are used': ['in'],
 'artificial intelligence': ['machine'],
 'as predictive': ['analytics'],
 'as subset': ['of'],
 'as training': ['data'],
 'being explicitly': ['programmed'],
 'build mathematical': ['model'],
 'business problems': ['machine'],
 'closely related': ['to'],
 'computational statistics': ['which'],
 'computer systems': ['use'],
 'computer vision': ['where'],
 'computers the': ['study'],
 'data analysis': ['through'],
 'data in': ['order'],
 'data known': ['as'],
 'data m

#### 1.3. Predict the next word:

In [ ]:
# Helper function that picks the following word.
def predict_next(a_nm1_gram):
    value_list_size = len(my_dict[a_nm1_gram])         # length of the value corresponding to the key = a_nm1_gram.
    i_pick = randint(0, value_list_size)               # A random number from the range 0 ~ value_list_size.
    return(my_dict[a_nm1_gram][i_pick])                  # Return the randomly chosen next word.

In [ ]:
# Test.
input_str = 'order to'                                 # Has to be a VALID (n-1)-Gram!
predict_next(input_str)

'make'

In [ ]:
# Another test.
# Repeat for 10 times and see that the next word is chosen randomly with a probability proportional to the occurrence.
input_str = 'machine learning'                                 # Has to be a VALID (n-1)-Gram!
for i in range(10):
    print(predict_next(input_str))

is
algorithms
is
data
data
is
algorithms
and
data
algorithms


#### 1.4. Predict a sequence:

In [ ]:
# Initialize the random seed.
seed(123)

In [ ]:
# A seed string has to be input by the user.
my_seed_str = 'machine learning'                                   # Has to be a VALID (n-1)-Gram!
# my_seed_str = 'in order'                                         # Has to be a VALID (n-1)-Gram!

In [ ]:
a_nm1_gram = my_seed_str
output_string = my_seed_str                                         # Initialize the output string.
while a_nm1_gram in my_dict:
    output_string += " " + predict_next(a_nm1_gram)
    words = nltk.word_tokenize(output_string)
    a_nm1_gram = ' '.join(words[-n+1:])                            # Update a_nm1_gram.

In [ ]:
# Output the predicted sequence.
output_string

'machine learning data mining is field of study within machine learning data mining is field of machine learning algorithms are used in the applications of email filtering detection of network intruders and computer vision where it is infeasible to develop an algorithm of specific instructions for performing the task machine learning and focuses on making predictions using computers the study of algorithms and statistical models that computer systems use to effectively perform specific task without using explicit instructions relying on patterns and inference instead it is seen as subset of artificial intelligence machine learning and focuses on exploratory data analysis through unsupervised learning in its application across business problems machine learning and focuses on exploratory data analysis through unsupervised learning in its application across business problems machine learning and focuses on exploratory data analysis through unsupervised learning in its application across 

In [ ]:
my_seed_str = 'in order'
output_string = my_seed_str                                         # Initialize the output string.
                                # Has to be a VALID (n-1)-Gram!
while a_nm1_gram in my_dict:
    output_string += " " + predict_next(a_nm1_gram)
    words = nltk.word_tokenize(output_string)
    a_nm1_gram = ' '.join(words[-n+1:])

In [ ]:
output_string

'in order'

In [ ]:
my_seed_str = 'learning is'
output_string = my_seed_str                                         # Initialize the output string.
                                # Has to be a VALID (n-1)-Gram!
while a_nm1_gram in my_dict:
    output_string += " " + predict_next(a_nm1_gram)
    words = nltk.word_tokenize(output_string)
    a_nm1_gram = ' '.join(words[-n+1:])

In [ ]:
output_string

'learning is'

# CountVectorizer en scikit-learn

Guía explicativa con ejemplos de uso. Basada en la documentación oficial de scikit-learn 1.9.1:
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

Todos los ejemplos fueron ejecutados con scikit-learn 1.9.1 y las salidas mostradas son las reales.

---

## 1. Qué hace

`CountVectorizer` convierte una colección de documentos de texto en una **matriz documento-término** de conteos. Cada fila es un documento, cada columna es un término (token) del vocabulario, y cada celda indica cuántas veces aparece ese término en ese documento.

Esta representación se conoce como **bolsa de palabras** (*bag of words*): se pierde el orden de las palabras y solo se conserva su frecuencia.

El proceso interno tiene tres etapas:

1. **Preprocesamiento**: pasar a minúsculas y, opcionalmente, quitar tildes.
2. **Tokenización**: separar el texto en tokens mediante una expresión regular.
3. **Generación de n-gramas**: construir unigramas, bigramas, etc., según se configure.

El resultado es una matriz dispersa (*sparse*) de SciPy, lo que permite manejar vocabularios grandes sin gastar memoria en los ceros.

```python
from sklearn.feature_extraction.text import CountVectorizer
```

---

## 2. Ejemplo básico

```python
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "El gato duerme en la casa.",
    "El perro juega en el parque.",
    "El gato y el perro juegan juntos.",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print(vectorizer.get_feature_names_out())
print(X.toarray())
```

Salida:

```
['casa' 'duerme' 'el' 'en' 'gato' 'juega' 'juegan' 'juntos' 'la' 'parque'
 'perro']
[[1 1 1 1 1 0 0 0 1 0 0]
 [0 0 2 1 0 1 0 0 0 1 1]
 [0 0 2 0 1 0 1 1 0 0 1]]
```

Observaciones:

- El vocabulario se ordena alfabéticamente.
- "El" y "el" se cuentan como el mismo término, porque `lowercase=True` por defecto.
- La letra "y" no aparece: el patrón de tokens por defecto solo acepta tokens de **2 o más caracteres**.
- "juega" y "juegan" son términos distintos: `CountVectorizer` no hace lematización ni stemming.
- La puntuación se descarta.

### Verlo como tabla con pandas

```python
import pandas as pd

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(df)
```

```
   casa  duerme  el  en  gato  juega  juegan  juntos  la  parque  perro
0     1       1   1   1     1      0       0       0   1       0      0
1     0       0   2   1     0      1       0       0   0       1      1
2     0       0   2   0     1      0       1       1   0       0      1
```

### La matriz es dispersa

```python
print(type(X), X.shape)
# <class 'scipy.sparse._csr.csr_matrix'> (3, 11)
```

Usar `.toarray()` solo con corpus pequeños. En corpus reales la matriz densa puede no caber en memoria.

---

## 3. Flujo fit / transform

El vocabulario se aprende con `fit` (o `fit_transform`) y luego se aplica a documentos nuevos con `transform`. Los términos que no estaban en el vocabulario aprendido se **ignoran**.

```python
nuevo = ["El gato juega con un raton"]
print(vectorizer.transform(nuevo).toarray())
# [[0 0 1 0 1 1 0 0 0 0 0]]
```

"con", "un" y "raton" no existían en el vocabulario de entrenamiento, por lo que no se cuentan.

Regla práctica: se hace `fit` solo sobre los datos de entrenamiento y `transform` sobre los de prueba, para no filtrar información del conjunto de prueba.

---

## 4. Parámetros principales

### 4.1 `ngram_range`: unigramas, bigramas y más

Tupla `(min_n, max_n)`. Por defecto `(1, 1)` (solo palabras sueltas). Con `(1, 2)` se agregan pares de palabras consecutivas, lo que captura algo de contexto.

```python
v = CountVectorizer(ngram_range=(1, 2))
v.fit(corpus)
print(v.get_feature_names_out())
```

```
['casa' 'duerme' 'duerme en' 'el' 'el gato' 'el parque' 'el perro' 'en'
 'en el' 'en la' 'gato' 'gato duerme' 'gato el' 'juega' 'juega en'
 'juegan' 'juegan juntos' 'juntos' 'la' 'la casa' 'parque' 'perro'
 'perro juega' 'perro juegan']
```

El bigrama "gato el" aparece porque la "y" fue descartada antes de formar los n-gramas. El vocabulario crece rápidamente al aumentar `max_n`.

### 4.2 `stop_words`: palabras vacías

Acepta `'english'` (lista incorporada solo para inglés) o una lista propia. Para español hay que entregar la lista manualmente (por ejemplo, desde NLTK o spaCy).

```python
v = CountVectorizer(stop_words=["el", "la", "en", "y"])
v.fit(corpus)
print(v.get_feature_names_out())
# ['casa' 'duerme' 'gato' 'juega' 'juegan' 'juntos' 'parque' 'perro']
```

La documentación advierte que la lista `'english'` tiene problemas conocidos y recomienda evaluar alternativas.

### 4.3 `min_df` y `max_df`: filtrar por frecuencia documental

- `min_df`: descarta términos que aparecen en **menos** documentos que el umbral. Útil para eliminar términos rarísimos o errores de tipeo.
- `max_df`: descarta términos que aparecen en **más** documentos que el umbral. Útil para eliminar palabras demasiado comunes del corpus, funcionando como stop words automáticas.

Si el valor es `int`, se interpreta como número de documentos; si es `float` entre 0.0 y 1.0, como proporción.

```python
v = CountVectorizer(min_df=2)          # aparece en al menos 2 documentos
v.fit(corpus)
print(v.get_feature_names_out())
# ['el' 'en' 'gato' 'perro']

v = CountVectorizer(max_df=0.9)        # aparece en a lo más el 90% de los documentos
v.fit(corpus)
print(v.get_feature_names_out())
# ['casa' 'duerme' 'en' 'gato' 'juega' 'juegan' 'juntos' 'la' 'parque'
#  'perro']
```

Con `max_df=0.9` se eliminó "el", que aparece en los 3 documentos (100%).

### 4.4 `max_features`: limitar el tamaño del vocabulario

Conserva solo los `k` términos más frecuentes en todo el corpus.

```python
v = CountVectorizer(max_features=3)
v.fit(corpus)
print(v.get_feature_names_out())
# ['el' 'en' 'gato']
```

Cuando hay empates en frecuencia ("en", "gato" y "perro" aparecen 2 veces cada uno), la selección entre los empatados no responde a un criterio semántico, por lo que conviene no depender de ese orden.

### 4.5 `binary`: presencia en lugar de conteo

Con `binary=True` todo conteo mayor que cero se convierte en 1. Es útil para modelos que trabajan con eventos binarios, como `BernoulliNB`.

```python
v = CountVectorizer(binary=True)
print(v.fit_transform(["el el el gato"]).toarray())
# [[1 1]]
```

### 4.6 `strip_accents`: normalizar tildes

Muy relevante en español. Acepta `'ascii'` (rápido, solo caracteres con equivalente ASCII directo) o `'unicode'` (más lento, funciona con cualquier carácter).

```python
v = CountVectorizer(strip_accents="unicode")
v.fit(["Educación pública y ciencia"])
print(v.get_feature_names_out())
# ['ciencia' 'educacion' 'publica']
```

Así, "educación" y "educacion" se tratan como el mismo término.

### 4.7 `analyzer`: palabras o caracteres

- `'word'` (por defecto): n-gramas de palabras.
- `'char'`: n-gramas de caracteres sobre todo el texto, incluidos los espacios.
- `'char_wb'`: n-gramas de caracteres solo dentro de cada palabra, rellenando los bordes con un espacio.
- Una función propia (callable) que reciba el texto crudo y devuelva la lista de features.

```python
v = CountVectorizer(analyzer="char_wb", ngram_range=(2, 3))
v.fit(["gato"])
print(v.get_feature_names_out())
# [' g' ' ga' 'at' 'ato' 'ga' 'gat' 'o ' 'to' 'to ']

v = CountVectorizer(analyzer="char", ngram_range=(2, 2))
v.fit(["gato perro"])
print(v.get_feature_names_out())
# [' p' 'at' 'er' 'ga' 'o ' 'pe' 'ro' 'rr' 'to']
```

Los n-gramas de caracteres son robustos frente a errores ortográficos y variaciones morfológicas, y sirven para tareas como identificación de idioma.

### 4.8 `token_pattern`: definir qué es un token

Expresión regular. La de defecto es `r"(?u)\b\w\w+\b"` (2 o más caracteres alfanuméricos). Para incluir palabras de una sola letra:

```python
v = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
v.fit(["y a o gato"])
print(v.get_feature_names_out())
# ['a' 'gato' 'o' 'y']
```

Si la expresión tiene un grupo de captura, el token será el contenido capturado (se permite como máximo un grupo).

### 4.9 `vocabulary`: vocabulario fijo

Se puede entregar un vocabulario predefinido. En ese caso no se aprende nada del corpus y se ignoran `min_df`, `max_df` y `max_features`.

```python
v = CountVectorizer(vocabulary=["gato", "perro", "casa"])
print(v.transform(corpus).toarray())
# [[1 0 1]
#  [0 1 0]
#  [1 1 0]]
```

Las columnas respetan el orden entregado. Como el vocabulario ya está definido, se puede llamar a `transform` sin `fit` previo. Esto es útil para garantizar que distintos experimentos usen exactamente las mismas columnas.

### 4.10 `preprocessor` y `tokenizer`: personalización

- `preprocessor`: reemplaza la etapa de preprocesamiento (minúsculas y tildes), manteniendo la tokenización y los n-gramas.
- `tokenizer`: reemplaza la tokenización, manteniendo el preprocesamiento y los n-gramas. Solo aplica con `analyzer='word'`.

```python
import re

def limpiar(texto):
    texto = texto.lower()
    texto = re.sub(r"\d+", "", texto)   # elimina numeros
    return texto

v = CountVectorizer(preprocessor=limpiar)
```

Al usar `preprocessor`, las opciones `lowercase` y `strip_accents` dejan de aplicarse automáticamente, por lo que la función propia debe encargarse de eso si se necesita.

### 4.11 Otros parámetros

| Parámetro | Defecto | Uso |
|---|---|---|
| `input` | `'content'` | `'content'` para strings, `'filename'` para rutas de archivos, `'file'` para objetos con método `read`. |
| `encoding` | `'utf-8'` | Codificación para decodificar bytes o archivos. |
| `decode_error` | `'strict'` | Qué hacer ante bytes inválidos: `'strict'` (error), `'ignore'` o `'replace'`. |
| `lowercase` | `True` | Pasar todo a minúsculas antes de tokenizar. |
| `dtype` | `np.int64` | Tipo de dato de la matriz resultante. |

---

## 5. Atributos después del ajuste

| Atributo | Descripción |
|---|---|
| `vocabulary_` | Diccionario que mapea cada término a su índice de columna. |
| `fixed_vocabulary_` | `True` si el vocabulario fue entregado por el usuario. |

```python
print(vectorizer.vocabulary_)
# {'el': 2, 'gato': 4, 'duerme': 1, 'en': 3, 'la': 8, 'casa': 0,
#  'perro': 10, 'juega': 5, 'parque': 9, 'juegan': 6, 'juntos': 7}
```

---

## 6. Métodos

| Método | Qué hace |
|---|---|
| `fit(raw_documents)` | Aprende el vocabulario. |
| `transform(raw_documents)` | Convierte documentos en matriz de conteos usando el vocabulario aprendido. |
| `fit_transform(raw_documents)` | Equivale a `fit` seguido de `transform`, pero más eficiente. |
| `get_feature_names_out()` | Devuelve los nombres de las columnas (términos). |
| `inverse_transform(X)` | Devuelve, por documento, los términos con conteo distinto de cero. |
| `build_analyzer()` | Devuelve la función completa de preprocesamiento, tokenización y n-gramas. |
| `build_preprocessor()` | Devuelve solo la función de preprocesamiento. |
| `build_tokenizer()` | Devuelve solo la función de tokenización. |
| `get_stop_words()` | Devuelve la lista efectiva de stop words. |
| `decode(doc)` | Decodifica la entrada a string unicode. |
| `get_params()` / `set_params()` | Leer o modificar hiperparámetros (útil en búsquedas de grilla). |

### `inverse_transform`

```python
print(vectorizer.inverse_transform(X[:1]))
# [array(['el', 'gato', 'duerme', 'en', 'la', 'casa'], dtype='<U6')]
```

### `build_analyzer`: depurar qué tokens se generan

Muy útil para verificar que la configuración produce lo esperado antes de vectorizar un corpus completo.

```python
analizador = CountVectorizer(ngram_range=(1, 2)).build_analyzer()
print(analizador("El Gato duerme"))
# ['el', 'gato', 'duerme', 'el gato', 'gato duerme']
```

---

## 7. Ejemplo completo: clasificación de textos con Pipeline

`CountVectorizer` suele ser el primer paso de un pipeline de clasificación. `MultinomialNB` funciona naturalmente con conteos.

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

textos = [
    "me encanta este curso",
    "excelente clase, muy clara",
    "la clase fue aburrida",
    "no me gusto la explicacion",
    "muy buena explicacion",
    "pesima organizacion del curso",
]
etiquetas = ["positivo", "positivo", "negativo",
             "negativo", "positivo", "negativo"]

modelo = make_pipeline(CountVectorizer(), MultinomialNB())
modelo.fit(textos, etiquetas)

print(modelo.predict(["la explicacion fue muy clara", "clase aburrida"]))
# ['positivo' 'negativo']
```

Al usar un pipeline, el `fit` del vectorizador ocurre solo con los datos de entrenamiento, lo que evita fugas de información en validación cruzada.

### Ajuste de hiperparámetros del vectorizador

```python
from sklearn.model_selection import GridSearchCV

parametros = {
    "countvectorizer__ngram_range": [(1, 1), (1, 2)],
    "countvectorizer__min_df": [1, 2],
    "countvectorizer__binary": [False, True],
}
busqueda = GridSearchCV(modelo, parametros, cv=3)
# busqueda.fit(textos_entrenamiento, etiquetas_entrenamiento)
```

Los parámetros de un componente del pipeline se nombran como `<componente>__<parametro>`.

---

## 8. Errores y cuidados frecuentes

1. **Hacer `fit` sobre el conjunto de prueba.** El vocabulario debe aprenderse solo con entrenamiento.
2. **Usar `.toarray()` en corpus grandes.** Puede agotar la memoria; la mayoría de los modelos de scikit-learn aceptan matrices dispersas directamente.
3. **Esperar que `stop_words='english'` sirva para español.** No sirve; hay que entregar una lista propia.
4. **Olvidar las palabras de una letra.** El patrón por defecto descarta "y", "a", "o", "e", "u".
5. **Suponer lematización.** "juega" y "juegan" son columnas distintas. Si se requiere, se debe pasar un `tokenizer` que lematice (por ejemplo, con spaCy).
6. **Tildes inconsistentes.** En textos en español conviene evaluar `strip_accents='unicode'`.
7. **Vocabulario explosivo con n-gramas.** Combinar `ngram_range` amplio con `min_df` o `max_features`.

---

## 9. Relación con otros vectorizadores

| Clase | Diferencia principal |
|---|---|
| `TfidfVectorizer` | Igual a `CountVectorizer` seguido de `TfidfTransformer`: pondera los conteos por TF-IDF, reduciendo el peso de términos muy comunes. |
| `TfidfTransformer` | Aplica TF-IDF sobre una matriz de conteos ya calculada. |
| `HashingVectorizer` | No guarda vocabulario; usa una función hash. Más eficiente en memoria y apto para flujos de datos, pero no permite recuperar los nombres de los términos. |

Cuándo preferir `CountVectorizer`:

- Cuando se necesitan conteos crudos (por ejemplo, para `MultinomialNB` o `LatentDirichletAllocation`).
- Cuando importa la interpretabilidad del vocabulario.
- Como base didáctica para entender la representación bolsa de palabras.

---

## 10. Resumen de la firma

```python
CountVectorizer(
    *,
    input="content",
    encoding="utf-8",
    decode_error="strict",
    strip_accents=None,
    lowercase=True,
    preprocessor=None,
    tokenizer=None,
    stop_words=None,
    token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 1),
    analyzer="word",
    max_df=1.0,
    min_df=1,
    max_features=None,
    vocabulary=None,
    binary=False,
    dtype=np.int64,
)
```

Todos los parámetros son de palabra clave (*keyword-only*), por lo que deben pasarse con su nombre.

In [ ]:
# Scrapear un libro/texto/sitio y con ese texto armar un modelo de lenguaje
# MODELO por ejemplo https://www.cervantesvirtual.com/obra-visor/el-ingenioso-hidalgo-don-quijote-de-la-mancha-6/html/05f86699-4b53-4d9b-8ab8-b40ab63fb0b3_2.html#I_5_
 # output[:200]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, f1_score

opiniones = resenas[resenas["nota"] != 3].copy()
opiniones["sentimiento"] = np.where(opiniones["nota"] >= 4, "positiva", "negativa")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Naive Bayes (conteos)": Pipeline([("bow", CountVectorizer()), ("modelo", MultinomialNB())]),
    "Logística (TF-IDF)": Pipeline([("tfidf", TfidfVectorizer()), ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000))]),
}
predicciones = {}
for nombre, modelo in modelos.items():
    predicciones[nombre] = cross_val_predict(modelo, opiniones["para_opinion"], opiniones["sentimiento"], cv=cv)
    print(f"===== {nombre}")
    print(classification_report(opiniones["sentimiento"], predicciones[nombre], digits=3))

**Qué deberías ver:** dos reportes con resultados parecidos. Comparen el **recall de la clase negativa** y la **precision**: cuando el modelo dice "negativa", ¿qué tan seguido acierta?, y de las negativas reales, ¿cuántas encuentra?

Un detalle importante: Naive Bayes no tiene `class_weight`. Como $P(\text{positiva})$ entra directo en la fórmula, con datos muy desbalanceados tiende a favorecer a la clase mayoritaria. Si trabajan con las reseñas reales, donde casi todas son positivas, es probable que lo noten.

### Ejercicio 4
Guarda en `mejor_f1_negativa` el **nombre** del modelo (una de las dos claves de `predicciones`) con mayor F1 en la clase negativa.

In [ ]:
mejor_f1_negativa = None  # TU CÓDIGO AQUÍ
print(mejor_f1_negativa)

In [ ]:
_f = {n: f1_score(opiniones["sentimiento"], p, pos_label="negativa") for n, p in predicciones.items()}
revisar(lambda: mejor_f1_negativa == max(_f, key=_f.get))

---
# Parte 4. Modelado de temas (láminas 135 a 154)

Tres formas de encontrar temas, de la más simple a la más elaborada:

| Método | Idea | Cada documento tiene |
|---|---|---|
| K-medias sobre TF-IDF | Agrupa documentos parecidos (Clase 11) | **Un** tema |
| LSA (análisis semántico latente) | SVD truncado de la matriz TF-IDF: cada componente es un tema | Una mezcla de temas (con pesos que pueden ser negativos) |
| LDA (asignación latente de Dirichlet) | Modelo probabilístico: cada documento es una mezcla de temas, y cada tema, una mezcla de palabras | Una **mezcla** de temas (probabilidades que suman 1) |

## Paso 4.1. LSA

In [ ]:
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

N_TEMAS = 4
terminos = vect.get_feature_names_out()

lsa = TruncatedSVD(n_components=N_TEMAS, random_state=42).fit(X_tfidf)
for i, comp in enumerate(lsa.components_):
    print(f"LSA tema {i}: {', '.join(terminos[np.argsort(comp)[::-1][:6]])}")

**Qué deberías ver:** 4 listas de palabras. El primer tema de LSA casi siempre reúne las palabras más comunes de todo el corpus (resume "de qué habla el corpus en general"); los siguientes separan temas más específicos. Es el mismo `TruncatedSVD` que usamos para graficar en 2D.

## Paso 4.2. LDA

LDA trabaja con **conteos** (como Naive Bayes), no con TF-IDF.

In [ ]:
vect_conteo = CountVectorizer(min_df=2)
X_conteo = vect_conteo.fit_transform(resenas["para_temas"])
terminos_c = vect_conteo.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=N_TEMAS, random_state=42).fit(X_conteo)
for i, comp in enumerate(lda.components_):
    print(f"LDA tema {i}: {', '.join(terminos_c[np.argsort(comp)[::-1][:6]])}")

mezcla = lda.transform(X_conteo)                        # cada fila: probabilidad de cada tema en esa reseña
print("\nPrimera reseña:", resenas.loc[0, "completo"][:90])
print("Mezcla de temas:", mezcla[0].round(2), "(suma:", round(mezcla[0].sum(), 2), ")")

**Qué deberías ver:** 4 temas con sus palabras y, para la primera reseña, 4 probabilidades que suman 1. Esa es la diferencia clave con K-medias: **una reseña puede hablar 60% de batería y 40% de pantalla**.

**Ojo:** con pocos documentos, LDA da temas inestables (cambian con `random_state`). Funciona bien desde algunos miles de documentos.

### Ejercicio 5
Guarda en `tema_principal_0` el número del tema de LDA con **mayor probabilidad** en la primera reseña (`mezcla[0]`).

In [ ]:
tema_principal_0 = None  # TU CÓDIGO AQUÍ
print(tema_principal_0)

In [ ]:
revisar(lambda: tema_principal_0 == int(np.argmax(mezcla[0])))

---
# Parte 5. Modelo de lenguaje con n-gramas (láminas 105 a 112)

> **Definición. Modelo de lenguaje:** un modelo que asigna una probabilidad a cada secuencia de palabras. Con eso puede **predecir la siguiente palabra**: es lo que hace el autocompletado del celular o del buscador (lámina 107).

Lo construimos paso a paso, primero a mano con un corpus de tres frases y después con código.

## Paso 5.1. El corpus de juguete

Tres frases. Agregamos dos marcas especiales: `<s>` al inicio y `</s>` al final, para que el modelo aprenda **cómo empiezan y cómo terminan** las frases.

| N.º | Frase con marcas |
|---|---|
| 1 | `<s>` me gusta el reloj `</s>` |
| 2 | `<s>` me gusta la pantalla `</s>` |
| 3 | `<s>` no me gusta la batería `</s>` |

## Paso 5.2. La regla de la cadena (lámina 108)

La probabilidad de una frase completa se puede escribir como una cadena de probabilidades condicionales: cada palabra, dado **todo lo anterior**.

$$P(\text{me gusta la pantalla}) = P(\text{me} \mid \text{<s>}) \cdot P(\text{gusta} \mid \text{<s> me}) \cdot P(\text{la} \mid \text{<s> me gusta}) \cdot \ldots$$

**El problema:** cuanto más larga la frase, más larga es la historia. "Dado `<s> me gusta la`" es una historia que casi nunca se repite exacta en los datos, así que no podemos contar cuántas veces sigue cada palabra.

## Paso 5.3. La simplificación: mirar solo las últimas palabras (láminas 110 y 111)

La **aproximación de Markov** dice: para predecir la siguiente palabra, basta con mirar las **últimas n − 1 palabras**, no toda la historia.

| Modelo | Mira | Probabilidad de "pantalla" en "me gusta la pantalla" |
|---|---|---|
| Unigrama (n = 1) | Nada: solo qué tan común es la palabra | P(pantalla) |
| **Bigrama (n = 2)** | **La palabra anterior** | **P(pantalla \| la)** |
| Trigrama (n = 3) | Las dos palabras anteriores | P(pantalla \| gusta la) |

Con un bigrama, la frase completa queda:

$$P(\text{me gusta la pantalla}) \approx P(\text{me} \mid \text{<s>}) \cdot P(\text{gusta} \mid \text{me}) \cdot P(\text{la} \mid \text{gusta}) \cdot P(\text{pantalla} \mid \text{la}) \cdot P(\text{</s>} \mid \text{pantalla})$$

## Paso 5.4. Cómo se calcula cada probabilidad: contando

$$P(\text{palabra} \mid \text{anterior}) = \frac{\text{veces que aparece "anterior palabra" juntas}}{\text{veces que aparece "anterior" seguida de algo}}$$

**A mano**, mirando las tres frases:

| Probabilidad | Conteo del par | Conteo de la anterior | Resultado |
|---|---|---|---|
| P(me \| `<s>`) | "`<s>` me" aparece **2** veces (frases 1 y 2) | `<s>` aparece **3** veces | 2/3 = 0,667 |
| P(gusta \| me) | "me gusta" aparece **3** veces | "me" aparece **3** veces | 3/3 = 1 |
| P(la \| gusta) | "gusta la" aparece **2** veces (frases 2 y 3) | "gusta" aparece **3** veces | 2/3 = 0,667 |
| P(el \| gusta) | "gusta el" aparece **1** vez | "gusta" aparece **3** veces | 1/3 = 0,333 |
| P(pantalla \| la) | "la pantalla" aparece **1** vez | "la" aparece **2** veces | 1/2 = 0,5 |
| P(`</s>` \| pantalla) | "pantalla `</s>`" aparece **1** vez | "pantalla" aparece **1** vez | 1/1 = 1 |

**La frase completa:**

$$P(\text{me gusta la pantalla}) = \tfrac{2}{3} \cdot 1 \cdot \tfrac{2}{3} \cdot \tfrac{1}{2} \cdot 1 = \tfrac{2}{9} \approx 0{,}222$$

## Paso 5.5. Cómo se predice la siguiente palabra

Para predecir qué sigue a "gusta", calculamos P(palabra | gusta) para **todas** las palabras posibles y elegimos la más alta:

- P(la | gusta) = 2/3
- P(el | gusta) = 1/3
- Cualquier otra = 0

La predicción es **"la"**. Esa es toda la "fórmula de predicción": **la palabra con mayor probabilidad condicional, dada la anterior**.

$$\text{predicción} = \arg\max_{w} \; P(w \mid \text{anterior})$$

($\arg\max$ significa "la palabra $w$ que hace más grande esta probabilidad".)

## Paso 5.6. Lo mismo con código

**Qué vamos a hacer:** contar los bigramas con `Counter` y comprobar los números de la tabla.

In [ ]:
frases = ["me gusta el reloj", "me gusta la pantalla", "no me gusta la batería"]

def con_marcas(frase):
    return ["<s>"] + frase.split() + ["</s>"]

bigramas = Counter()        # cuántas veces aparece cada par (anterior, palabra)
anteriores = Counter()      # cuántas veces aparece cada palabra como "anterior" de algo
for frase in frases:
    tokens = con_marcas(frase)
    for anterior, palabra in zip(tokens[:-1], tokens[1:]):   # zip arma los pares de palabras seguidas
        bigramas[(anterior, palabra)] += 1
        anteriores[anterior] += 1

print("Pares contados:", dict(bigramas))
print("\nAnteriores:", dict(anteriores))

**Qué deberías ver:** por ejemplo `('me', 'gusta'): 3`, `('gusta', 'la'): 2` y `('gusta', 'el'): 1`; y en anteriores, `'gusta': 3`.

`zip(tokens[:-1], tokens[1:])` empareja cada palabra con la siguiente: con `[<s>, me, gusta]` arma `(<s>, me)` y `(me, gusta)`.

Ahora la probabilidad, la predicción y la probabilidad de una frase completa:

In [ ]:
def prob(palabra, anterior):
    """P(palabra | anterior) = conteo del par / conteo de la anterior."""
    if anteriores[anterior] == 0:
        return 0.0
    return bigramas[(anterior, palabra)] / anteriores[anterior]

def predecir_siguiente(anterior):
    """Todas las palabras que siguieron alguna vez a 'anterior', con su probabilidad, de mayor a menor."""
    candidatas = {p: prob(p, a) for (a, p) in bigramas if a == anterior}
    return sorted(candidatas.items(), key=lambda x: -x[1])

def prob_frase(frase):
    tokens = con_marcas(frase)
    total = 1.0
    for anterior, palabra in zip(tokens[:-1], tokens[1:]):
        p = prob(palabra, anterior)
        print(f"  P({palabra} | {anterior}) = {p:.3f}")
        total *= p
    return total

print("P(la | gusta) =", round(prob("la", "gusta"), 3))
print("Después de 'gusta' viene:", predecir_siguiente("gusta"))
print("\nP('me gusta la pantalla'):")
print("  Total =", round(prob_frase("me gusta la pantalla"), 3))

**Qué deberías ver:** exactamente los números de la tabla a mano: 0,667 para "la" después de "gusta", y 0,222 para la frase completa.

### Ejercicio 6
Calcula **a mano** (con fracciones) la probabilidad de la frase **"no me gusta la pantalla"** y guárdala en `p_frase`. Pista: son 6 factores, empezando por P(no | `<s>`). Después compruébala con `prob_frase`.

In [ ]:
p_frase = None  # TU CÓDIGO AQUÍ, por ejemplo (1/3) * ...
print(p_frase)

In [ ]:
revisar(lambda: np.isclose(p_frase, 1/3 * 1 * 1 * 2/3 * 1/2 * 1))

## Paso 5.7. El problema de los ceros y el suavizado

¿Qué probabilidad tiene "me gusta el pantalla"? El par "el pantalla" **nunca** apareció, así que P(pantalla | el) = 0, y **toda la frase vale 0**. Con un corpus real, la mayoría de las frases nuevas tienen algún par nunca visto.

> **Definición. Suavizado de Laplace (sumar uno):** sumar 1 a todos los conteos, para que ningún par tenga probabilidad cero.
>
> $$P_{\text{Laplace}}(\text{palabra} \mid \text{anterior}) = \frac{\text{conteo del par} + 1}{\text{conteo de la anterior} + V}$$
>
> donde **V** es el número de palabras distintas que pueden venir después (el vocabulario, incluido `</s>`). Se suma V abajo para que las probabilidades sigan sumando 1.

En nuestro corpus V = 9 (me, gusta, el, la, reloj, pantalla, no, batería, `</s>`). Entonces:

$$P_{\text{Laplace}}(\text{pantalla} \mid \text{el}) = \frac{0 + 1}{1 + 9} = 0{,}1$$

In [ ]:
VOCAB = {p for (_, p) in bigramas}             # todas las palabras que aparecen después de algo (incluye </s>)
V = len(VOCAB)

def prob_laplace(palabra, anterior):
    return (bigramas[(anterior, palabra)] + 1) / (anteriores[anterior] + V)

print("V =", V)
print(f"P(pantalla | el): sin suavizado = {prob('pantalla', 'el'):.3f} | con Laplace = {prob_laplace('pantalla', 'el'):.3f}")
print(f"P(la | gusta):    sin suavizado = {prob('la', 'gusta'):.3f} | con Laplace = {prob_laplace('la', 'gusta'):.3f}")

**Qué deberías ver:** "pantalla después de el" pasa de 0 a 0,1, y "la después de gusta" baja de 0,667 a 0,25. El suavizado le "quita" un poco de probabilidad a lo que sí vimos para dársela a lo que no vimos.

## Paso 5.8. Un autocompletado con las reseñas

**Qué vamos a hacer:** el mismo modelo, ahora con **trigramas** (mira las dos palabras anteriores) y con las reseñas como corpus.

**Importante:** para un modelo de lenguaje **no quitamos stopwords ni hacemos stemming**. Queremos predecir texto real, y "de", "la" o "que" son justo las palabras que más se predicen.

In [ ]:
def tokens_simples(texto):
    return re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower()).split()

tri, ctx = Counter(), Counter()
for texto in resenas["completo"]:
    for oracion in re.split(r"[.!?]", texto):          # cada oración por separado
        t = ["<s>", "<s>"] + tokens_simples(oracion) + ["</s>"]
        if len(t) < 4:
            continue
        for a, b, c in zip(t[:-2], t[1:-1], t[2:]):      # tríos de palabras seguidas
            tri[(a, b, c)] += 1
            ctx[(a, b)] += 1

def siguiente(a, b, n=3):
    """Las n palabras más probables después de 'a b', con su probabilidad."""
    opciones = {c: tri[(x, y, c)] / ctx[(a, b)] for (x, y, c) in tri if (x, y) == (a, b)}
    return sorted(opciones.items(), key=lambda x: -x[1])[:n]

for inicio in [("la", "batería"), ("la", "pantalla"), ("no", "me")]:
    print(" ".join(inicio), "→", [(p, round(v, 2)) for p, v in siguiente(*inicio)])

**Qué deberías ver:** las palabras que más veces siguieron a cada par en las reseñas, con su probabilidad. Usamos dos `<s>` al inicio porque un trigrama necesita dos palabras anteriores incluso para la primera palabra.

Y si vamos eligiendo siempre la más probable, el modelo **genera texto**:

In [ ]:
def generar(a="<s>", b="<s>", maximo=15):
    palabras = []
    for _ in range(maximo):
        opciones = siguiente(a, b, n=1)
        if not opciones or opciones[0][0] == "</s>":
            break
        palabra = opciones[0][0]
        palabras.append(palabra)
        a, b = b, palabra                                # la ventana avanza una palabra
    return " ".join(palabras)

print("Desde el inicio:   ", generar())
print("Desde 'la batería':", "la batería", generar("la", "batería"))

**Qué deberías ver:** frases que "suenan" a reseña, armadas con trozos de las reseñas reales. Elegir siempre la más probable se llama **búsqueda voraz** (*greedy*), y tiende a repetir lo más común.

**El puente con el Capítulo 10:** ChatGPT y los demás modelos de lenguaje grandes hacen **lo mismo en esencia**: predicen la siguiente palabra (en realidad, el siguiente token) dada la historia. La diferencia es que en vez de contar pares en 60 reseñas, usan una red neuronal enorme que mira miles de palabras hacia atrás y aprendió de billones de textos.

### Ejercicio 7
Con el corpus de juguete, ¿qué palabra predice el modelo de **bigramas** después de **"la"**? Hay un empate: guarda en `empate_la` una **lista ordenada alfabéticamente** con las palabras empatadas.

In [ ]:
empate_la = None  # TU CÓDIGO AQUÍ
print(empate_la)

In [ ]:
revisar(lambda: empate_la == ["batería", "pantalla"])

---
# Parte 6. Skip-gram: vectores de palabras (láminas 159 a 167)

El modelo de n-gramas existe para **predecir**. Skip-gram (Word2Vec) también predice, pero solo como excusa: lo que queremos son **vectores** (embeddings) donde palabras de significado parecido queden cerca. Resuelve el problema del one-hot y de TF-IDF: "celular" y "teléfono" deberían quedar cerca aunque sean palabras distintas.

## Paso 6.1. Los pares de entrenamiento

Skip-gram toma cada palabra como **central** y trata de predecir las palabras de alrededor, dentro de una **ventana**. Con la frase de la lámina 157 y una ventana de 2:

In [ ]:
frase = "como una manzana todas las mañanas".split()
VENTANA = 2

pares = []
for i, central in enumerate(frase):
    for j in range(max(0, i - VENTANA), min(len(frase), i + VENTANA + 1)):
        if j != i:
            pares.append((central, frase[j]))

print(f"{len(pares)} pares (central, contexto):")
for p in pares:
    print(" ", p)

**Qué deberías ver:** pares como `('manzana', 'como')`, `('manzana', 'una')`, `('manzana', 'todas')` y `('manzana', 'las')`. La red aprende a predecir el segundo elemento a partir del primero, y al hacerlo ajusta un vector para cada palabra.

**CBOW** (la otra variante) hace lo contrario: con las palabras de alrededor, predice la central.

### Ejercicio 8
¿Cuántos pares se generan con la misma frase y una **ventana de 1**? Guárdalo en `n_pares_v1`.

In [ ]:
n_pares_v1 = None  # TU CÓDIGO AQUÍ
print(n_pares_v1)

In [ ]:
revisar(lambda: n_pares_v1 == 10)

## Paso 6.2. Entrenar Word2Vec con las reseñas

**Qué vamos a hacer:** entrenar skip-gram (`sg=1`) con `gensim` sobre las oraciones de las reseñas, y preguntar qué palabras quedan más cerca de algunas palabras clave.

In [ ]:
from gensim.models import Word2Vec

oraciones = [tokens_simples(o) for texto in resenas["completo"] for o in re.split(r"[.!?]", texto)]
oraciones = [o for o in oraciones if len(o) > 2]

w2v = Word2Vec(oraciones, vector_size=50, window=3, min_count=2, sg=1, epochs=200, seed=42, workers=1)
print("Palabras con vector:", len(w2v.wv))
print("Vector de 'batería' (primeros 8 de 50 valores):", w2v.wv["batería"][:8].round(2))

for palabra in ["batería", "pantalla", "celular"]:
    if palabra in w2v.wv:
        print(f"\nMás parecidas a '{palabra}':", [(p, round(s, 2)) for p, s in w2v.wv.most_similar(palabra, topn=5)])

**Qué deberías ver:** cada palabra tiene ahora un vector de 50 números, y una lista de palabras "parecidas" según la similitud coseno entre vectores.

**Seamos honestos con el resultado:** con unos cientos de oraciones, algunas palabras parecidas tendrán sentido (palabras que aparecen en las mismas frases) y otras no. Word2Vec necesita **millones** de palabras para que los vectores capturen bien el significado. Por eso, en la práctica, se usan vectores **ya entrenados** con grandes corpus, o modelos como los del Capítulo 10, que ya traen esa representación.

---
# Cierre

| Parte | Lo esencial |
|---|---|
| Codificación entera | Cada palabra a un número; las más frecuentes, los números más bajos. 0 para relleno y 1 para desconocidas. |
| Relleno | Todas las secuencias del mismo largo, el formato que necesitan las redes neuronales. |
| Similitud coseno | Compara la dirección de dos vectores; no le importa el largo del documento. |
| Naive Bayes | Probabilidad de la clase por las probabilidades de cada palabra en esa clase. |
| LSA y LDA | Temas como componentes (LSA) o como mezclas de probabilidad (LDA). |
| N-gramas | P(palabra \| anterior) = conteo del par / conteo de la anterior. Se predice la de mayor probabilidad. |
| Laplace | Sumar 1 para que nada tenga probabilidad cero. |
| Skip-gram | Predecir el contexto para aprender vectores de palabras. |

Guarden el notebook en `ejercicios_estudiantes` como `clase12_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1

In [ ]:
palabra_top = frecuencias.most_common(1)[0][0]
print(palabra_top, palabra_a_indice[palabra_top])

### Ejercicio 2

In [ ]:
X_20 = rellenar(secuencias, 20)
print(X_20.shape, "| ceros:", (X_20 == 0).sum())

### Ejercicio 3
B · D = 1·2 + 1·2 + 0·1 = 4. |B| = √2. |D| = √9 = 3. cos = 4 / (√2 · 3) = 0,943.

In [ ]:
cos_bd = B @ D / (np.linalg.norm(B) * np.linalg.norm(D))
print(round(cos_bd, 3))

### Ejercicio 4

In [ ]:
_f1 = {n: f1_score(opiniones["sentimiento"], p, pos_label="negativa") for n, p in predicciones.items()}
mejor_f1_negativa = max(_f1, key=_f1.get)
print(mejor_f1_negativa, {n: round(v, 3) for n, v in _f1.items()})

### Ejercicio 5

In [ ]:
tema_principal_0 = int(np.argmax(mezcla[0]))
print(tema_principal_0)

### Ejercicio 6
P(no | `<s>`) = 1/3 · P(me | no) = 1 · P(gusta | me) = 1 · P(la | gusta) = 2/3 · P(pantalla | la) = 1/2 · P(`</s>` | pantalla) = 1. Total = 1/9 ≈ 0,111.

In [ ]:
p_frase = (1/3) * 1 * 1 * (2/3) * (1/2) * 1
print(round(p_frase, 3), "| con la función:", round(prob_frase("no me gusta la pantalla"), 3))

### Ejercicio 7
"la" aparece 2 veces como anterior: una vez seguida de "pantalla" y otra de "batería". Las dos tienen probabilidad 1/2.

In [ ]:
opciones = predecir_siguiente("la")
maximo = opciones[0][1]
empate_la = sorted(p for p, v in opciones if v == maximo)
print(opciones, "→", empate_la)

### Ejercicio 8
Con ventana 1, cada palabra se empareja solo con sus vecinas inmediatas. Las 2 palabras de los extremos tienen 1 vecina y las 4 del medio tienen 2: 2·1 + 4·2 = 10.

In [ ]:
n_pares_v1 = sum(1 for i in range(len(frase)) for j in range(max(0, i - 1), min(len(frase), i + 2)) if j != i)
print(n_pares_v1)